In [ ]:
"""
Scenario 1 -- Global stability of the coexistence equilibrium E* 
"""
import numpy as np
import pickle
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D
from mpl_toolkits.mplot3d import
from scipy.integrate import solve_ivp

DAY = 1.0
MONTH = 30.0
YEAR = 365.0

P = dict(
    r=0.18, K=5e8, k1=7.09e-9, alpha1=11.0, alpha2=0.02, alpha3=0.08,
    alpha4=1e-4, delta1=2.019e7, delta2=5e7, Lambda_E=1.3e4, Lambda_R=1e3,
    xi=3.422e-10, dE=0.033, dR=0.10, uc=0.5, beta=1e-6, eta=0.10,
    zetaP=0.02,                     
    delta_b=0.01,                    
    Lambda_P=500.0,                  
)
P["E0star"] = P["Lambda_E"] / P["dE"]  

theta_deltaP, k_deltaP = 61.1 * MONTH, 0.297
theta_lambda, k_lambda = 67.9 * MONTH, 1.213
h_max, tau_h = 0.02, 90.0

tau_pic = 7.0
g_max = 0.889
kappa_max = 1.11e-6
n_hill = 1
delta_e0, delta_e_max, sigma_e = 0.0293, 0.161, 10.0
Gamma_max, sigma_Treg = 3.5e-5, 15.0


def h_fun(tau):
    tau = np.asarray(tau, dtype=float)
    return h_max * (tau / tau_h) * np.exp(1 - tau / tau_h)


def deltaP_fun(tau):
    tau = np.asarray(tau, dtype=float)
    return (k_deltaP / theta_deltaP) * (tau / theta_deltaP) ** (k_deltaP - 1)


def lambda_fun(tau):
    tau = np.asarray(tau, dtype=float)
    return (k_lambda / theta_lambda) * (tau / theta_lambda) ** (k_lambda - 1)


def g_fun(sigma):
    sigma = np.asarray(sigma, dtype=float)
    return g_max / (1 + (sigma / tau_pic) ** (8 * n_hill))


def kappa_fun(sigma):
    sigma = np.asarray(sigma, dtype=float)
    return kappa_max / (1 + (sigma / tau_pic) ** (8 * n_hill))


def deltae_fun(sigma):
    sigma = np.asarray(sigma, dtype=float)
    return delta_e0 + (delta_e_max - delta_e0) / (1 + (sigma / sigma_e) ** n_hill)


def Gamma_fun(sigma):
    sigma = np.asarray(sigma, dtype=float)
    return Gamma_max / (1 + (sigma / sigma_Treg) ** n_hill)


def Theta(X, alpha, R, uc):
    denom = X + alpha * (1 - uc) * R
    if np.isscalar(X):
        return 0.0 if X <= 0 else X / denom
    out = np.zeros_like(np.asarray(X, dtype=float))
    mask = np.asarray(X) > 0
    out[mask] = np.asarray(X)[mask] / (denom[mask] if hasattr(denom, "__len__") else denom)
    return out


def Phi(C, delta):
    return C / (delta + C)

def make_grids(tau_max=6 * YEAR, Ntau=150, sigma_max=2 * MONTH, Nsigma=60):
    dtau = tau_max / Ntau
    tau_mid = (np.arange(Ntau) + 0.5) * dtau
    dsigma = sigma_max / Nsigma
    sigma_mid = (np.arange(Nsigma) + 0.5) * dsigma
    return dtau, tau_mid, dsigma, sigma_mid


class M2Model:

    def __init__(self, psi_fun, phi_fun=None, Ntau=150, Nsigma=60,
                 tau_max=6 * YEAR, sigma_max=2 * MONTH):
        self.dtau, self.tau, self.dsigma, self.sigma = make_grids(tau_max, Ntau, sigma_max, Nsigma)
        self.Ntau, self.Nsigma = Ntau, Nsigma
        self.psi = psi_fun
        self.phi = phi_fun if phi_fun is not None else (lambda t: P["Lambda_P"])
        self.h_t = h_fun(self.tau)
        self.dP_t = deltaP_fun(self.tau)
        self.lam_t = lambda_fun(self.tau)
        self.g_s = g_fun(self.sigma)
        self.kap_s = kappa_fun(self.sigma)
        self.de_s = deltae_fun(self.sigma)
        self.Ga_s = Gamma_fun(self.sigma)

    def unpack(self, y):
        C, E, R = y[0], y[1], y[2]
        Pv = y[3:3 + self.Ntau]
        Mv = y[3 + self.Ntau:3 + self.Ntau + self.Nsigma]
        return C, E, R, Pv, Mv

    def unpack_series(self, Y):
        C, E, R = Y[0], Y[1], Y[2]
        Pt = Y[3:3 + self.Ntau]
        Mt = Y[3 + self.Ntau:3 + self.Ntau + self.Nsigma]
        return C, E, R, Pt, Mt

    def rhs(self, t, y):
        pr = P
        C, E, R, Pv, Mv = self.unpack(y)
        C = max(C, 0.0); E = max(E, 0.0); R = max(R, 0.0)
        Pv = np.clip(Pv, 0, None); Mv = np.clip(Mv, 0, None)

        omega = np.sum(self.lam_t * Pv) * self.dtau
        muM = np.sum(self.kap_s * Mv) * self.dsigma
        NM = np.sum(Mv) * self.dsigma

        ThetaE = Theta(E, pr["alpha1"], R, pr["uc"])
        ThetaN = Theta(NM, pr["alpha1"], R, pr["uc"])
        Theta1 = Theta(1.0, pr["alpha4"], R, pr["uc"])
        Phi_C_d1 = Phi(C, pr["delta1"])
        Phi_C_beta = Phi(C, 1.0 / pr["beta"])
        Phi_C_d2 = Phi(C, pr["delta2"])

        kill_E = pr["k1"] * (1 - Phi_C_beta) * ThetaE * E * C
        kill_M = ThetaN * muM * C

        dC = omega + pr["r"] * C * (1 - C / pr["K"]) - kill_E - kill_M
        dE = (pr["Lambda_E"] + pr["alpha2"] * Theta1 * Phi_C_d1 * E - pr["xi"] * E * C - pr["dE"] * E
              + pr["eta"] * (kill_E + kill_M))
        dR = pr["Lambda_R"] + pr["alpha3"] * Phi_C_d1 * R - pr["dR"] * R

        phit = self.phi(t)
        src_P = self.h_t - self.dP_t - self.lam_t - pr["zetaP"] * E / pr["E0star"]
        dPv = np.empty(self.Ntau)
        dPv[0] = (phit - Pv[0]) / self.dtau + src_P[0] * Pv[0]
        dPv[1:] = (Pv[:-1] - Pv[1:]) / self.dtau + src_P[1:] * Pv[1:]

        psit = self.psi(t)
        src_M = self.g_s * Phi_C_d2 - pr["delta_b"] - self.de_s - self.Ga_s * (1 - pr["uc"]) * R
        dMv = np.empty(self.Nsigma)
        dMv[0] = (psit - Mv[0]) / self.dsigma + src_M[0] * Mv[0]
        dMv[1:] = (Mv[:-1] - Mv[1:]) / self.dsigma + src_M[1:] * Mv[1:]

        return np.concatenate(([dC, dE, dR], dPv, dMv))

    def initial_state(self, C0=1e6, E0=None, R0=None):
        pr = P
        E0 = pr["Lambda_E"] / pr["dE"] if E0 is None else E0
        R0 = pr["Lambda_R"] / pr["dR"] if R0 is None else R0
        Pv0 = (pr["Lambda_P"] / max(self.dP_t[0], 1e-6)) * np.exp(-self.dP_t * self.tau / 40.0)
        Pv0 = np.clip(Pv0, 0, None)
        Mv0 = np.zeros(self.Nsigma)
        return np.concatenate(([C0, E0, R0], Pv0, Mv0))

    def solve(self, t_span, t_eval, y0=None, method="BDF"):
        if y0 is None:
            y0 = self.initial_state()
        return solve_ivp(self.rhs, t_span, y0, t_eval=t_eval, method=method,
                          rtol=1e-6, atol=1e-3, max_step=5.0)

    def nonlocal_series(self, sol):
        C, E, R, Pt, Mt = self.unpack_series(sol.y)
        omega = np.einsum("i,it->t", self.lam_t, Pt) * self.dtau
        muM = np.einsum("i,it->t", self.kap_s, Mt) * self.dsigma
        NM = Mt.sum(axis=0) * self.dsigma
        return omega, muM, NM

class M1Model:
    def __init__(self, mu_fun, N_fun, lam_eff, deltaP_eff, phi_fun=None):
        self.mu = mu_fun
        self.N = N_fun
        self.lam_eff = lam_eff
        self.deltaP_eff = deltaP_eff
        self.phi = phi_fun if phi_fun is not None else (lambda t: P["Lambda_P"])

    def rhs(self, t, y):
        pr = P
        Pp, C, E, R = y
        Pp = max(Pp, 0.0); C = max(C, 0.0); E = max(E, 0.0); R = max(R, 0.0)
        mu_t = self.mu(t); N_t = self.N(t)

        ThetaE = Theta(E, pr["alpha1"], R, pr["uc"])
        ThetaN = Theta(N_t, pr["alpha1"], R, pr["uc"])
        Theta1 = Theta(1.0, pr["alpha4"], R, pr["uc"])
        Phi_C_d1 = Phi(C, pr["delta1"])
        Phi_C_beta = Phi(C, 1.0 / pr["beta"])

        kill_E = pr["k1"] * (1 - Phi_C_beta) * ThetaE * E * C
        kill_M = ThetaN * mu_t * C

        dP = self.phi(t) - self.deltaP_eff * Pp - pr["zetaP"] * E / pr["E0star"] * Pp - self.lam_eff * Pp
        dC = self.lam_eff * Pp + pr["r"] * C * (1 - C / pr["K"]) - kill_E - kill_M
        dE = (pr["Lambda_E"] + pr["alpha2"] * Theta1 * Phi_C_d1 * E - pr["xi"] * E * C - pr["dE"] * E
              + pr["eta"] * (kill_E + kill_M))
        dR = pr["Lambda_R"] + pr["alpha3"] * Phi_C_d1 * R - pr["dR"] * R
        return np.array([dP, dC, dE, dR])

    def initial_state(self, C0=1e6, P0=1e5, E0=None, R0=None):
        pr = P
        E0 = pr["Lambda_E"] / pr["dE"] if E0 is None else E0
        R0 = pr["Lambda_R"] / pr["dR"] if R0 is None else R0
        return np.array([P0, C0, E0, R0])

    def solve(self, t_span, t_eval, y0=None, method="BDF"):
        if y0 is None:
            y0 = self.initial_state()
        return solve_ivp(self.rhs, t_span, y0, t_eval=t_eval, method=method,
                          rtol=1e-7, atol=1e-2, max_step=5.0)

def healthy_M_profile(sigma, psi_ref, Rs, uc):
    Ga = Gamma_fun(sigma)
    de = deltae_fun(sigma)
    integrand = P["delta_b"] + de + Ga * (1 - uc) * Rs
    cum = np.concatenate(([0], np.cumsum((integrand[:-1] + integrand[1:]) / 2 * np.diff(sigma))))
    return psi_ref * np.exp(-cum)


def compute_R0_Rg(psi_ref, sigma_max=2 * MONTH, Ns=4000, C_omega=None):
    pr = P
    Es = pr["Lambda_E"] / pr["dE"]
    Rs = pr["Lambda_R"] / pr["dR"]
    sigma = np.linspace(1e-3, sigma_max, Ns)
    Ms = healthy_M_profile(sigma, psi_ref, Rs, pr["uc"])
    Nbar = np.trapezoid(Ms, sigma)
    mubar = np.trapezoid(kappa_fun(sigma) * Ms, sigma)

    alpha1p = pr["alpha1"] * (1 - pr["uc"])
    term_E = pr["k1"] * Es ** 2 / (Es + alpha1p * Rs)
    term_M = (Nbar * mubar) / (Nbar + alpha1p * Rs) if Nbar > 0 else 0.0
    D = term_E + term_M
    R0 = pr["r"] / D if D > 0 else np.inf

    if C_omega is None:
        C_omega = pr["K"]
    mE = pr["Lambda_E"] / (pr["dE"] + pr["xi"] * C_omega)
    MR = pr["Lambda_R"] / (pr["dR"] - pr["alpha3"])
    term_E_g = (pr["k1"] / (1 + pr["beta"] * C_omega)) * (mE ** 2 / (mE + alpha1p * MR))
    mu_star = mubar
    term_M_g = (mu_star ** 2) / (mu_star + kappa_max * alpha1p * MR) if mu_star > 0 else 0.0
    Dg = term_E_g + term_M_g
    Rg = pr["r"] / Dg if Dg > 0 else np.inf
    return dict(Es=Es, Rs=Rs, D=D, R0=R0, Dg=Dg, Rg=Rg, Nbar=Nbar, mubar=mubar)

PSI_CONST = 1.5e4          

print("=== Step 1: running M2 to steady state under constant ACT ===")
psi_c = lambda t: PSI_CONST
m2 = M2Model(psi_c)
t_eval2 = np.linspace(0, 20 * YEAR, 200)
sol2 = m2.solve((0, 20 * YEAR), t_eval2, y0=m2.initial_state(C0=1e7))
C2, E2, R2, Pt2, Mt2 = m2.unpack_series(sol2.y)
omega2, muM2, NM2 = m2.nonlocal_series(sol2)
Cstar2, Estar2, Rstar2 = C2[-1], E2[-1], R2[-1]
omega_star2, mu_star2, N_star2 = omega2[-1], muM2[-1], NM2[-1]
print(f"  E*_2 = ({Cstar2:.4e}, {Estar2:.4e}, {Rstar2:.4e})")

th2 = compute_R0_Rg(psi_ref=PSI_CONST, C_omega=P["K"])
R0_2 = th2["R0"]
print(f"  R0_2 = {R0_2:.4f}")

print("=== Step 2: calibrating M1 (mu1, N1 DERIVED from M2's own disease-free "
      "profile -- shared parameters only, no hand-tuned constant) ===")
alpha1p = P["alpha1"] * (1 - P["uc"])
Es_h = P["Lambda_E"] / P["dE"]
Rs_h = P["Lambda_R"] / P["dR"]
N_M1, mu_M1 = th2["Nbar"], th2["mubar"]
A_const = P["k1"] * Es_h ** 2 / (Es_h + alpha1p * Rs_h)
ThetaN_h = Theta(N_M1, P["alpha1"], Rs_h, P["uc"])

tau_ref = np.linspace(30, 1800, 2000)
lam_eff = lambda_fun(tau_ref).mean()
deltaP_eff = deltaP_fun(tau_ref).mean()

m1 = M1Model(lambda t: mu_M1, lambda t: N_M1, lam_eff, deltaP_eff)
t_eval1 = np.linspace(0, 20 * YEAR, 200)
sol1 = m1.solve((0, 20 * YEAR), t_eval1, y0=m1.initial_state(C0=4e8))
Pp1, C1, E1, R1 = sol1.y
Cstar1, Estar1, Rstar1 = C1[-1], E1[-1], R1[-1]
omega_star1 = lam_eff * Pp1[-1]

D1 = A_const + ThetaN_h * mu_M1
R0_1 = P["r"] / D1
print(f"  mu_M1={mu_M1:.4e}  N_M1={N_M1:.4e}")
print(f"  E*_1 = ({Cstar1:.4e}, {Estar1:.4e}, {Rstar1:.4e})")
print(f"  R0_1 = {R0_1:.4f}   (R0_2 = {R0_2:.4f} -- equal by construction)")
print(f"  balance check: C1*/C2* = {Cstar1/Cstar2:.2f}")

def reduced_rhs(t, y, omega, muM, NM):
    pr = P
    C, E, R = y
    C = max(C, 0.0); E = max(E, 0.0); R = max(R, 0.0)
    ThetaE = Theta(E, pr["alpha1"], R, pr["uc"])
    ThetaN = Theta(NM, pr["alpha1"], R, pr["uc"])
    Theta1 = Theta(1.0, pr["alpha4"], R, pr["uc"])
    Phi_C_d1 = Phi(C, pr["delta1"])
    Phi_C_beta = Phi(C, 1.0 / pr["beta"])
    kill_E = pr["k1"] * (1 - Phi_C_beta) * ThetaE * E * C
    kill_M = ThetaN * muM * C
    dC = omega + pr["r"] * C * (1 - C / pr["K"]) - kill_E - kill_M
    dE = (pr["Lambda_E"] + pr["alpha2"] * Theta1 * Phi_C_d1 * E - pr["xi"] * E * C - pr["dE"] * E
          + pr["eta"] * (kill_E + kill_M))
    dR = pr["Lambda_R"] + pr["alpha3"] * Phi_C_d1 * R - pr["dR"] * R
    return np.array([dC, dE, dR])


print("=== Step 3: sanity check (reduced RHS should vanish at each equilibrium) ===")
print("  residual at E*_1:", reduced_rhs(0, [Cstar1, Estar1, Rstar1], omega_star1, mu_M1, N_M1))
print("  residual at E*_2:", reduced_rhs(0, [Cstar2, Estar2, Rstar2], omega_star2, mu_star2, N_star2))

print("=== Step 4: integrating 15 initial conditions for each model ===")
rng = np.random.default_rng(7)
n_ic = 15
C0s = np.exp(rng.uniform(np.log(2e6), np.log(4.2e8), n_ic))
E0s = np.exp(rng.uniform(np.log(1e5), np.log(2.2e8), n_ic))
R0s = np.exp(rng.uniform(np.log(2e3), np.log(1.1e5), n_ic))


def integrate_ics(omega, muM, NM, t_max=8 * YEAR):
    trajs = []
    for i in range(n_ic):
        y0 = [C0s[i], E0s[i], R0s[i]]
        sol = solve_ivp(reduced_rhs, (0, t_max), y0, args=(omega, muM, NM),
                         method="BDF", rtol=1e-7, atol=1e-2, max_step=5.0,
                         t_eval=np.linspace(0, t_max, 400))
        trajs.append(sol.y)
    return trajs


trajs1 = integrate_ics(omega_star1, mu_M1, N_M1)
trajs2 = integrate_ics(omega_star2, mu_star2, N_star2)

plt.rcParams.update({
    "font.size": 10, "axes.titlesize": 10.5, "axes.labelsize": 10,
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.grid": True, "grid.alpha": 0.2,
})

SC, SE, SR = 1e8, 1e7, 1e4
scale = {"C": SC, "E": SE, "R": SR}
labels = {"C": r"$C\;(\times 10^{8})$", "E": r"$E\;(\times 10^{7})$", "R": r"$R\;(\times 10^{4})$"}


def reduced_rhs_grid(C, E, R, omega, muM, NM):
    pr = P
    C = np.clip(C, 0, None); E = np.clip(E, 0, None); R = np.clip(R, 0, None)
    ThetaE = Theta(E, pr["alpha1"], R, pr["uc"])
    ThetaN = Theta(NM, pr["alpha1"], R, pr["uc"])
    Theta1 = Theta(1.0, pr["alpha4"], R, pr["uc"])
    Phi_C_d1 = Phi(C, pr["delta1"])
    Phi_C_beta = Phi(C, 1.0 / pr["beta"])
    kill_E = pr["k1"] * (1 - Phi_C_beta) * ThetaE * E * C
    kill_M = ThetaN * muM * C
    dC = omega + pr["r"] * C * (1 - C / pr["K"]) - kill_E - kill_M
    dE = (pr["Lambda_E"] + pr["alpha2"] * Theta1 * Phi_C_d1 * E - pr["xi"] * E * C - pr["dE"] * E
          + pr["eta"] * (kill_E + kill_M))
    dR = pr["Lambda_R"] + pr["alpha3"] * Phi_C_d1 * R - pr["dR"] * R
    return dC, dE, dR


def stream_panel(ax, xk, yk, fixed, omega, muM, NM, xmax, ymax, cmap):
    xs_raw = np.linspace(1e3, xmax, 90)
    ys_raw = np.linspace(1e3, ymax, 90)
    XX, YY = np.meshgrid(xs_raw, ys_raw)
    vals = dict(fixed)
    vals[xk] = XX; vals[yk] = YY
    dC, dE, dR = reduced_rhs_grid(vals["C"], vals["E"], vals["R"], omega, muM, NM)
    dvals = {"C": dC, "E": dE, "R": dR}
    U, V = dvals[xk] / scale[xk], dvals[yk] / scale[yk]
    xs, ys = xs_raw / scale[xk], ys_raw / scale[yk]
    mag = np.sqrt(U ** 2 + V ** 2)
    magn = np.log10(mag + 1.0)
    magn = (magn - magn.min()) / (magn.max() - magn.min() + 1e-30)
    ax.streamplot(xs, ys, U, V, color=magn, cmap=cmap, density=1.25,
                  linewidth=1.1, arrowsize=1.0, norm=plt.Normalize(0, 1), zorder=1)


def add_trajectories(ax, trajs, xk, yk, color_list):
    idx = {"C": 0, "E": 1, "R": 2}
    for i, tr in enumerate(trajs):
        x = tr[idx[xk]] / scale[xk]
        y = tr[idx[yk]] / scale[yk]
        ax.plot(x, y, color=color_list[i], lw=1.6, alpha=0.95, zorder=4,
                path_effects=[pe.Stroke(linewidth=2.6, foreground="white"), pe.Normal()])
        ax.scatter([x[0]], [y[0]], color=color_list[i], s=22, zorder=5,
                   edgecolor="k", linewidth=0.4)


colors = plt.cm.tab20(np.linspace(0, 1, n_ic))

fig = plt.figure(figsize=(17.5, 9.6))
gs = fig.add_gridspec(2, 4, wspace=0.16, hspace=0.30, width_ratios=[1, 1, 1, 1.35],
                       left=0.075, right=0.99, top=0.89, bottom=0.06)

rows = [
    dict(name="Model 1 (unstructured model)", eq=(Cstar1, Estar1, Rstar1),
         omega=omega_star1, mu=mu_M1, N=N_M1, R0=R0_1, trajs=trajs1, cmap="cool"),
    dict(name="Model 2 (structured model)", eq=(Cstar2, Estar2, Rstar2),
         omega=omega_star2, mu=mu_star2, N=N_star2, R0=R0_2, trajs=trajs2, cmap="cool"),
]

panel_defs = [("C", "E", "R"), ("C", "R", "E"), ("E", "R", "C")]
panel_letters = ["a)", "b)", "c)", "d)", "e)", "f)", "g)", "h)"]
 
maxs = {
    "C": max(C0s.max(), Cstar1, Cstar2) * 1.25,
    "E": max(E0s.max(), Estar1, Estar2) * 1.25,
    "R": max(R0s.max(), Rstar1, Rstar2) * 1.25,
}

for i, row in enumerate(rows):
    Cs, Es, Rs = row["eq"]
    fixed_vals = {"C": Cs, "E": Es, "R": Rs}

    axes_row = []
    for j, (xk, yk, fk) in enumerate(panel_defs):
        ax = fig.add_subplot(gs[i, j])
        axes_row.append(ax)

        stream_panel(ax, xk, yk, dict(fixed_vals), row["omega"], row["mu"], row["N"],
                     maxs[xk], maxs[yk], row["cmap"])
        add_trajectories(ax, row["trajs"], xk, yk, colors)

        xe, ye = fixed_vals[xk] / scale[xk], fixed_vals[yk] / scale[yk]
        ax.plot([xe], [ye], marker="*", color="gold", markeredgecolor="black",
                markersize=20, zorder=6)

        ax.set_xlim(0, maxs[xk] / scale[xk])
        ax.set_ylim(0, maxs[yk] / scale[yk])
        ax.set_xlabel(labels[xk], fontsize=11)
        ax.set_ylabel(labels[yk], fontsize=11)
        letter = panel_letters[i * 4 + j]
        ax.set_title(f"{letter} ({xk},{yk}) plane at {fk}={fk}$^*$", fontsize=12.5, pad=5)
        ax.tick_params(labelsize=9.5)
 
        if i == 0 and j == 0:
            ax.legend(
                [Line2D([0], [0], marker="*", color="w", markerfacecolor="gold",
                        markeredgecolor="black", markersize=14, lw=0),
                 Line2D([0], [0], color="cyan", lw=2),
                 Line2D([0], [0], color="tab:orange", lw=1.2)],
                ["coexistence equilibrium $\\mathcal{E}^*$",
                 "vector field (streamlines)", "sample trajectory"],
                loc="upper right", fontsize=8.3, framealpha=0.9,
            )

    axes_row[0].text(-0.24, 0.5, f"{row['name']}\n" + r"$\mathcal{R}_0^{(%d)}=%.3f$" % (i + 1, row["R0"]),
                      transform=axes_row[0].transAxes, rotation=90, va="center", ha="center",
                      fontsize=13, fontweight="bold")

    ax3 = fig.add_subplot(gs[i, 3], projection="3d")
    for k, tr in enumerate(row["trajs"]):
        ax3.plot(tr[0] / SC, tr[1] / SE, tr[2] / SR, color=colors[k], lw=1.0, alpha=0.85)
        ax3.scatter([tr[0, 0] / SC], [tr[1, 0] / SE], [tr[2, 0] / SR], color=colors[k], s=12)
    ax3.scatter([Cs / SC], [Es / SE], [Rs / SR], marker="*", color="gold",
                edgecolor="black", s=220, zorder=6)
    ax3.set_xlabel("C ($\\times10^8$)", labelpad=6, fontsize=10.5)
    ax3.set_ylabel("E ($\\times10^7$)", labelpad=6, fontsize=10.5)
    ax3.set_zlabel("R ($\\times10^4$)", labelpad=-2, fontsize=10.5)
    ax3.set_xlim(0, maxs["C"] / SC)
    ax3.set_ylim(0, maxs["E"] / SE)
    ax3.set_zlim(0, maxs["R"] / SR)
    letter = panel_letters[i * 4 + 3]
    ax3.set_title(f"{letter} 3D phase portrait", fontsize=12.5, pad=3)
    ax3.view_init(elev=18, azim=-120)  
    ax3.set_box_aspect(None, zoom=1.25)
    ax3.tick_params(labelsize=8.5)

cax = fig.add_axes([0.30, 0.955, 0.22, 0.018])
sm = plt.cm.ScalarMappable(cmap="cool", norm=plt.Normalize(0, 1))
sm.set_array([])
cbar = fig.colorbar(sm, cax=cax, orientation="horizontal")
cbar.set_label("streamline colour: normalized flow speed (slow $\\to$ fast)", fontsize=9)
cbar.ax.tick_params(labelsize=7.5)
cbar.set_ticks([0, 0.5, 1])
cbar.set_ticklabels(["slow", "", "fast"])

fig.savefig("./fig1_global_stability_en.png", dpi=175, bbox_inches="tight")
print("\nSaved fig1_global_stability_en.png")
print(f"E*_1 = ({Cstar1:.4e}, {Estar1:.4e}, {Rstar1:.4e})   R0_1 = {R0_1:.4f}")
print(f"E*_2 = ({Cstar2:.4e}, {Estar2:.4e}, {Rstar2:.4e})   R0_2 = {R0_2:.4f}")






"""
Scenario -- fixed tau_pic=6 days, three doses (columns), comparing a
SINGLE ACT campaign (n_c=1) against TWO campaigns 75 days apart.
"""

import numpy as np
import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
from matplotlib.colors import PowerNorm
from scipy.integrate import solve_ivp

MONTH = 30.0
YEAR = 365.0

P = dict(
    r=0.18,
    K=5e8,
    k1=7.09e-9,
    alpha1=11.0,
    alpha2=0.02,
    alpha3=0.08,
    alpha4=1e-4,
    delta1=2.019e7,
    delta2=5e7,
    Lambda_E=1.3e4,
    Lambda_R=1e3,
    xi=3.422e-10,
    dE=0.033,
    dR=0.10,
    uc=0.5,
    beta=1e-6,
    eta=0.10,
    zetaP=0.02,
    delta_b=0.01,
    Lambda_P=500.0,
)

P["E0star"] = P["Lambda_E"] / P["dE"]


theta_deltaP = 61.1 * MONTH
k_deltaP = 0.297

theta_lambda = 67.9 * MONTH
k_lambda = 1.213

h_max = 0.02
tau_h = 90.0

tau_pic = 7.0

g_max = 0.889
kappa_max = 1.11e-6

n_hill = 1

delta_e0 = 0.0293
delta_e_max = 0.161
sigma_e = 10.0

Gamma_max = 1e-5
sigma_Treg = 15.0


def h_fun(tau):

    tau = np.asarray(
        tau,
        dtype=float
    )

    return (
        h_max
        * (tau / tau_h)
        * np.exp(
            1 - tau / tau_h
        )
    )


def deltaP_fun(tau):

    tau = np.asarray(
        tau,
        dtype=float
    )

    return (
        (k_deltaP / theta_deltaP)
        * (tau / theta_deltaP)
        ** (k_deltaP - 1)
    )


def lambda_fun(tau):

    tau = np.asarray(
        tau,
        dtype=float
    )

    return (
        (k_lambda / theta_lambda)
        * (tau / theta_lambda)
        ** (k_lambda - 1)
    )


def g_fun(sigma):

    sigma = np.asarray(
        sigma,
        dtype=float
    )

    return (
        g_max
        /
        (
            1
            + (
                sigma / tau_pic
            ) ** (8 * n_hill)
        )
    )


def kappa_fun(sigma):

    sigma = np.asarray(
        sigma,
        dtype=float
    )

    return (
        kappa_max
        /
        (
            1
            + (
                sigma / tau_pic
            ) ** (8 * n_hill)
        )
    )


def deltae_fun(sigma):

    sigma = np.asarray(
        sigma,
        dtype=float
    )

    return (
        delta_e0
        +
        (delta_e_max - delta_e0)
        /
        (
            1
            + (
                sigma / sigma_e
            ) ** n_hill
        )
    )


def Gamma_fun(sigma):

    sigma = np.asarray(
        sigma,
        dtype=float
    )

    return (
        Gamma_max
        /
        (
            1
            + (
                sigma / sigma_Treg
            ) ** n_hill
        )
    )
 

def Theta(
    X,
    alpha,
    R,
    uc
):

    denom = (
        X
        + alpha
        * (1 - uc)
        * R
    )

    if np.isscalar(X):

        if X <= 0:

            return 0.0

        return X / denom

    out = np.zeros_like(
        np.asarray(
            X,
            dtype=float
        )
    )

    mask = (
        np.asarray(X) > 0
    )

    if hasattr(
        denom,
        "__len__"
    ):

        out[mask] = (
            np.asarray(X)[mask]
            / denom[mask]
        )

    else:

        out[mask] = (
            np.asarray(X)[mask]
            / denom
        )

    return out


def Phi(
    C,
    delta
):

    return C / (
        delta + C
    )


# =======================================================================
# 4. Grids
# =======================================================================

def make_grids(
    tau_max,
    Ntau,
    sigma_max,
    Nsigma
):

    dtau = (
        tau_max / Ntau
    )

    tau_mid = (
        np.arange(Ntau)
        + 0.5
    ) * dtau

    dsigma = (
        sigma_max / Nsigma
    )

    sigma_mid = (
        np.arange(Nsigma)
        + 0.5
    ) * dsigma

    return (
        dtau,
        tau_mid,
        dsigma,
        sigma_mid
    )
 

class M2Model:

    def __init__(
        self,
        psi_fun,
        phi_fun=None,
        Ntau=200,
        Nsigma=60,
        tau_max=11 * YEAR,
        sigma_max=40.0
    ):

        (
            self.dtau,
            self.tau,
            self.dsigma,
            self.sigma
        ) = make_grids(
            tau_max,
            Ntau,
            sigma_max,
            Nsigma
        )

        self.Ntau = Ntau
        self.Nsigma = Nsigma

        self.psi = psi_fun

        if phi_fun is not None:

            self.phi = phi_fun

        else:

            self.phi = (
                lambda t:
                P["Lambda_P"]
            )

        self.h_t = h_fun(
            self.tau
        )

        self.dP_t = deltaP_fun(
            self.tau
        )

        self.lam_t = lambda_fun(
            self.tau
        )

        self.g_s = g_fun(
            self.sigma
        )

        self.kap_s = kappa_fun(
            self.sigma
        )

        self.de_s = deltae_fun(
            self.sigma
        )

        self.Ga_s = Gamma_fun(
            self.sigma
        )


    def unpack_series(
        self,
        Y
    ):

        C = Y[0]
        E = Y[1]
        R = Y[2]

        Pt = Y[
            3:
            3 + self.Ntau
        ]

        Mt = Y[
            3 + self.Ntau:
            3
            + self.Ntau
            + self.Nsigma
        ]

        return (
            C,
            E,
            R,
            Pt,
            Mt
        )


    def rhs(
        self,
        t,
        y
    ):

        pr = P

        C = y[0]
        E = y[1]
        R = y[2]

        Pv = y[
            3:
            3 + self.Ntau
        ]

        Mv = y[
            3 + self.Ntau:
            3
            + self.Ntau
            + self.Nsigma
        ]

        C = max(
            C,
            0.0
        )

        E = max(
            E,
            0.0
        )

        R = max(
            R,
            0.0
        )

        Pv = np.clip(
            Pv,
            0,
            None
        )

        Mv = np.clip(
            Mv,
            0,
            None
        )

        omega = (
            np.sum(
                self.lam_t
                * Pv
            )
            * self.dtau
        )

        muM = (
            np.sum(
                self.kap_s
                * Mv
            )
            * self.dsigma
        )

        NM = (
            np.sum(Mv)
            * self.dsigma
        )


        ThetaE = Theta(
            E,
            pr["alpha1"],
            R,
            pr["uc"]
        )

        ThetaN = Theta(
            NM,
            pr["alpha1"],
            R,
            pr["uc"]
        )

        Theta1 = Theta(
            1.0,
            pr["alpha4"],
            R,
            pr["uc"]
        )


        Phi_C_d1 = Phi(
            C,
            pr["delta1"]
        )

        Phi_C_beta = Phi(
            C,
            1.0 / pr["beta"]
        )

        Phi_C_d2 = Phi(
            C,
            pr["delta2"]
        )


        kill_E = (
            pr["k1"]
            * (
                1
                - Phi_C_beta
            )
            * ThetaE
            * E
            * C
        )

        kill_M = (
            ThetaN
            * muM
            * C
        )

        dC = (
            omega
            + pr["r"]
            * C
            * (
                1
                - C / pr["K"]
            )
            - kill_E
            - kill_M
        )

        dE = (
            pr["Lambda_E"]
            + pr["alpha2"]
            * Theta1
            * Phi_C_d1
            * E
            - pr["xi"]
            * E
            * C
            - pr["dE"]
            * E
            + pr["eta"]
            * (
                kill_E
                + kill_M
            )
        )



        dR = (
            pr["Lambda_R"]
            + pr["alpha3"]
            * Phi_C_d1
            * R
            - pr["dR"]
            * R
        )

        phit = self.phi(t)

        src_P = (
            self.h_t
            - self.dP_t
            - self.lam_t
            - pr["zetaP"]
            * E
            / pr["E0star"]
        )

        dPv = np.empty(
            self.Ntau
        )

        dPv[0] = (
            (
                phit
                - Pv[0]
            )
            / self.dtau
            + src_P[0]
            * Pv[0]
        )

        dPv[1:] = (
            (
                Pv[:-1]
                - Pv[1:]
            )
            / self.dtau
            + src_P[1:]
            * Pv[1:]
        )

        psit = self.psi(t)

        src_M = (
            self.g_s
            * Phi_C_d2
            - pr["delta_b"]
            - self.de_s
            - self.Ga_s
            * (
                1
                - pr["uc"]
            )
            * R
        )

        dMv = np.empty(
            self.Nsigma
        )

        dMv[0] = (
            (
                psit
                - Mv[0]
            )
            / self.dsigma
            + src_M[0]
            * Mv[0]
        )

        dMv[1:] = (
            (
                Mv[:-1]
                - Mv[1:]
            )
            / self.dsigma
            + src_M[1:]
            * Mv[1:]
        )


        return np.concatenate(
            (
                [dC, dE, dR],
                dPv,
                dMv
            )
        )


    def initial_state(
        self,
        C0
    ):

        E0 = (
            P["Lambda_E"]
            / P["dE"]
        )

        R0 = (
            P["Lambda_R"]
            / P["dR"]
        )

        return np.concatenate(
            (
                [C0, E0, R0],
                np.zeros(
                    self.Ntau
                ),
                np.zeros(
                    self.Nsigma
                )
            )
        )


    def solve(
        self,
        t_span,
        t_eval,
        y0,
        method="BDF"
    ):

        return solve_ivp(
            self.rhs,
            t_span,
            y0,
            t_eval=t_eval,
            method=method,
            rtol=1e-6,
            atol=1e-3,
            max_step=5.0
        )


DI = 7.0

D_VALUES = [
    100000.0,
    150000.0,
    200000.0
]

T_START = 3 * MONTH

CAMPAIGN_GAP = 75.0

T_MAX = 1 * YEAR


def make_psi(
    dose,
    n_camp,
    di=DI
):

    if n_camp == 1:

        camp_times = [
            T_START
        ]

    else:

        camp_times = [
            T_START,
            T_START
            + CAMPAIGN_GAP
        ]


    def psi(t):

        for ts in camp_times:

            if (
                ts <= t
                < ts + di
            ):

                return dose / di

        return 0.0


    return psi

TAU_PIC = 6.0
N_EXPONENT = 1.0

M_SCALE = 1e4


def run(
    D,
    n_camp
):

    m2 = M2Model(
        make_psi(
            D,
            n_camp
        ),

        Nsigma=150,

        tau_max=
        T_MAX + 60.0,

        sigma_max=80.0
    )
    m2.g_s = (
        0.889
        /
        (
            1
            + (
                m2.sigma
                / TAU_PIC
            )
            ** N_EXPONENT
        )
    )


    m2.kap_s = (
        1.11e-6
        /
        (
            1
            + (
                m2.sigma
                / TAU_PIC
            )
            ** N_EXPONENT
        )
    )


    t_eval = np.linspace(
        0,
        T_MAX,
        550
    )


    sol = m2.solve(
        (0, T_MAX),
        t_eval,
        m2.initial_state(
            P["K"]
        )
    )


    C, E, R, Pt, Mt = (
        m2.unpack_series(
            sol.y
        )
    )


    return (
        t_eval,
        C,
        Mt,
        m2.sigma
    )

print(
    f"Running tau_pic={TAU_PIC}, "
    f"n={N_EXPONENT}, "
    f"comparing 1 vs 2 campaigns "
    f"(gap={CAMPAIGN_GAP:.0f} d) "
    f"across D={D_VALUES} ..."
)


results = {}


for D in D_VALUES:

    for n_camp in [1, 2]:

        t_eval, C, Mt, sigma = run(
            D,
            n_camp
        )


        results[
            (D, n_camp)
        ] = {

            "t": t_eval,

            "C": C,

            "Mt": Mt,

            "sigma": sigma
        }


        floor_pct = (
            100
            * C.min()
            / P["K"]
        )


        print(
            f"  D={D:.0f}, "
            f"n_c={n_camp}: "
            f"floor={floor_pct:.1f}% of K"
        )

def sci_fmt(x):

    if x == 0:

        return "0"


    exp = int(
        np.floor(
            np.log10(
                abs(x)
            )
        )
    )


    mant = (
        x
        / 10 ** exp
    )


    if abs(
        mant
        - round(mant)
    ) < 1e-9:

        mant_str = (
            f"{mant:.0f}"
        )

    else:

        mant_str = (
            f"{mant:.1f}"
        )


    return (
        f"${mant_str}"
        f"\\times10^{{{exp}}}$"
    )

plt.rcParams.update({

    "font.family":
        "DejaVu Sans",

    "font.size":
        12,

    "axes.titlesize":
        13,

    "axes.labelsize":
        12.5,

    "figure.facecolor":
        "white",

    "axes.facecolor":
        "white",

    "axes.grid":
        True,

    "grid.alpha":
        0.30,

    "grid.linewidth":
        0.6,
})


panel_letters = [
    f"{chr(97+i)})"
    for i in range(9)
]


COL_1CAMP = "#2465A8"
COL_2CAMP = "#D14B3B"

def polished_surface(
    ax,
    X,
    Y,
    Z,
    cmap_name,
    vmin,
    vmax
):

    cmap = plt.get_cmap(
        cmap_name
    )


    norm = PowerNorm(
        gamma=0.5,
        vmin=vmin,
        vmax=vmax
    )


    ax.plot_surface(
        X,
        Y,
        Z,

        cmap=cmap,

        norm=norm,

        rstride=2,

        cstride=2,

        linewidth=0,

        antialiased=True
    )


    for axis in [
        ax.xaxis,
        ax.yaxis,
        ax.zaxis
    ]:

        axis.pane.set_edgecolor(
            "none"
        )

        axis.pane.set_alpha(
            0.04
        )


    ax.grid(False)


    sm = plt.cm.ScalarMappable(
        cmap=cmap,
        norm=norm
    )

    sm.set_array([])


    return sm

Mt1_all = [
    results[
        (D, 1)
    ]["Mt"]
    for D in D_VALUES
]


Mt2_all = [
    results[
        (D, 2)
    ]["Mt"]
    for D in D_VALUES
]


vmax1 = max(
    m.max()
    for m in Mt1_all
)


vmax2 = max(
    m.max()
    for m in Mt2_all
)

fig = plt.figure(
    figsize=(17, 15.5)
)

gs_top = fig.add_gridspec(
    1,
    3,

    wspace=0.16,

    left=0.055,
    right=0.895,

    top=0.965,
    bottom=0.735
)

gs_bot = fig.add_gridspec(
    2,
    3,

    wspace=-0.05,

    hspace=0.30,

    left=0.075,
    right=0.875,

    top=0.675,
    bottom=0.075
)


axes_top = []
axes_mid = []
axes_bot = []


for j, D in enumerate(
    D_VALUES
):

    r1 = results[
        (D, 1)
    ]

    r2 = results[
        (D, 2)
    ]


    t_months = (
        r1["t"]
        / MONTH
    )


    ax = fig.add_subplot(
        gs_top[0, j]
    )

    axes_top.append(
        ax
    )


    ax.plot(
        t_months,

        r2["C"] / 1e8,

        color=COL_2CAMP,

        lw=1.8,

        ls="-",

        label="$n_c=2$ injections",

        zorder=4
    )


    ax.plot(
        t_months,

        r1["C"] / 1e8,

        color=COL_1CAMP,

        lw=1.8,

        ls="--",

        label="$n_c=1$ injection",

        zorder=5
    )


    ax.axhline(
        P["K"] / 1e8,

        color="0.4",

        lw=1.2,

        zorder=2
    )


    ax.text(
        0.3,

        P["K"] / 1e8 * 1.025,

        "$C=K$",

        fontsize=9.5,

        color="0.35"
    )


    ax.set_ylim(
        -P["K"] / 1e8 * 0.03,

        P["K"] / 1e8 * 1.08
    )


    ax.set_yticks(
        np.arange(0, 6, 1)
    )


    ax.set_xlim(
        0,
        12
    )


    ax.set_xlabel(
        "time (months)"
    )


    if j == 0:

        ax.set_ylabel(
            "$C(t)$ "
            "($\\times10^{8}$ cells)"
        )


        ax.legend(
            loc="lower right",

            fontsize=9.5,

            framealpha=0.92
        )


    ax.set_title(
        f"{panel_letters[j]}  "
        f"$D=${sci_fmt(D)}",

        fontsize=12.5,

        pad=8
    )


    ax.spines[
        ["top", "right"]
    ].set_visible(False)


    ax3d_mid = fig.add_subplot(
        gs_bot[0, j],
        projection="3d"
    )

    axes_mid.append(
        ax3d_mid
    )


    ax3d_mid.set_facecolor(
        "none"
    )


    T_grid, S_grid = np.meshgrid(
        r1["t"] / MONTH,

        r1["sigma"],

        indexing="ij"
    )


    sm1 = polished_surface(
        ax3d_mid,

        S_grid,

        T_grid,

        r1["Mt"].T
        / M_SCALE,

        "viridis",

        0,

        vmax1 / M_SCALE
    )


    ax3d_mid.set_xlabel(
        "Time since injection $\\sigma$ (days)",

        fontsize=8.5,

        labelpad=5
    )


    ax3d_mid.set_ylabel(
        "time (months)",

        fontsize=8.5,

        labelpad=5
    )


    ax3d_mid.set_zlabel(
        "$M(t,\\sigma)$ "
        "($\\times10^{4}$)",

        fontsize=8.5,

        labelpad=3
    )



    ax3d_mid.text2D(
        0.50,
        1.035,

        f"{panel_letters[3+j]}",

        transform=ax3d_mid.transAxes,

        ha="center",

        va="bottom",

        fontsize=12.5
    )


    ax3d_mid.view_init(
        elev=35,
        azim=-125
    )


    ax3d_mid.set_box_aspect(
        None,

        zoom=0.90
    )


    ax3d_mid.tick_params(
        labelsize=7,

        pad=2
    )

    ax3d_bot = fig.add_subplot(
        gs_bot[1, j],
        projection="3d"
    )

    axes_bot.append(
        ax3d_bot
    )


    ax3d_bot.set_facecolor(
        "none"
    )


    T_grid, S_grid = np.meshgrid(
        r2["t"] / MONTH,

        r2["sigma"],

        indexing="ij"
    )


    sm2 = polished_surface(
        ax3d_bot,

        S_grid,

        T_grid,

        r2["Mt"].T
        / M_SCALE,

        "plasma",

        0,

        vmax2 / M_SCALE
    )


    ax3d_bot.set_xlabel(
        "Time since injection $\\sigma$ (days)",

        fontsize=8.5,

        labelpad=5
    )


    ax3d_bot.set_ylabel(
        "time (months)",

        fontsize=8.5,

        labelpad=5
    )


    ax3d_bot.set_zlabel(
        "$M(t,\\sigma)$ "
        "($\\times10^{4}$)",

        fontsize=8.5,

        labelpad=3
    )
    ax3d_bot.text2D(
        0.50,
        1.035,

        f"{panel_letters[6+j]}",

        transform=ax3d_bot.transAxes,

        ha="center",

        va="bottom",

        fontsize=12.5
    )


    ax3d_bot.view_init(
        elev=35,
        azim=-125
    )


    ax3d_bot.set_box_aspect(
        None,

        zoom=0.90
    )


    ax3d_bot.tick_params(
        labelsize=7,

        pad=2
    )


row2_bbox = [
    ax.get_position()
    for ax in axes_mid
]


row3_bbox = [
    ax.get_position()
    for ax in axes_bot
]


y2_lo = min(
    b.y0
    for b in row2_bbox
)


y2_hi = max(
    b.y1
    for b in row2_bbox
)


y3_lo = min(
    b.y0
    for b in row3_bbox
)


y3_hi = max(
    b.y1
    for b in row3_bbox
)


x_left = min(
    b.x0
    for b in row2_bbox
)


fig.text(
    x_left - 0.045,

    (y2_lo + y2_hi) / 2,

    "One injection",

    rotation=90,

    va="center",

    ha="center",

    fontsize=13
)


fig.text(
    x_left - 0.045,

    (y3_lo + y3_hi) / 2,

    "Two injections",

    rotation=90,

    va="center",

    ha="center",

    fontsize=13
)

cax1 = fig.add_axes(
    [
        0.905,

        y2_lo,

        0.012,

        y2_hi - y2_lo
    ]
)


cbar1 = fig.colorbar(
    sm1,

    cax=cax1
)


cbar1.ax.tick_params(
    labelsize=7.5
)


cbar1.set_label(
    "$M(t,\\sigma)$ "
    "($\\times10^{4}$)",

    fontsize=9
)


cax2 = fig.add_axes(
    [
        0.905,

        y3_lo,

        0.012,

        y3_hi - y3_lo
    ]
)


cbar2 = fig.colorbar(
    sm2,

    cax=cax2
)


cbar2.ax.tick_params(
    labelsize=7.5
)


cbar2.set_label(
    "$M(t,\\sigma)$ "
    "($\\times10^{4}$)",

    fontsize=9
)

output_file = (
    "./fig_injections_1vs2_final_nouveau_joel_en3.png"
)


fig.savefig(
    output_file,

    bbox_inches="tight",

    dpi=185
)


plt.close(fig)


print(
    f"Saved {output_file}"
)
























"""
Scenario -- comparing M1 (unstructured/mean-field) and M2 (structured/
age-resolved) models on the "Efficacy" metric (1-year cumulative
tumour-burden reduction, single ACT campaign), across tau_pic, D, and
u_c. For each panel, one parameter is varied over discrete bins while
the other two are drawn at random (Latin-hypercube-like uniform
sampling) within their reference ranges, giving a DISTRIBUTION of
efficacy values per bin -- shown as paired boxplots (M1 vs M2 side by
side) with the interquartile range and whiskers acting as an implicit
confidence interval on the comparison at each bin.

M1's constant ACT parameters (mu_M1, N_M1) are calibrated from M2's
own coupled simulation (age-structured M(t,sigma) together with the
real C(t) feedback) at the SAME (tau_pic, D, u_c), time-averaged over
the post-injection window -- see compare_core.py for the full
rationale. This is the fairest available comparison given M1 has no
native tau_pic/age-structure parameter.

Output: fig_compare_M1M2_final_nouveau_joel_en.png
"""
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# =======================================================================
# NOTE: this script is fully standalone. The content of compare_core.py
# (calibration + efficacy functions for M1 and M2, which itself inlines
# core.py's M2Model/parameter definitions) has been merged in directly
# below, so no external file is required to run this script.
# =======================================================================

"""
Core functions for comparing M1 (unstructured/mean-field) and M2
(structured/age-resolved) models on the "Efficacy" metric (1-year
cumulative tumour-burden reduction, single ACT campaign), used by
scenario_compare_M1M2_final.py.

M1's constant ACT parameters (mu, N) are calibrated from M2's OWN
disease-free ACT profile M_s(sigma) at the same (tau_pic, D, u_c),
using a sustained-rate proxy psi_const = D/DI (the campaign's
injection rate) -- exactly the calibration principle used in the
"global stability" scenario (Scenario 1) to make M1 and M2 comparable
at matched R0, rather than picking arbitrary M1 constants.
"""
import numpy as np
from scipy.integrate import solve_ivp

"""
Scenario 4 (honest version) -- Cumulative tumour burden: Model 1 (ODE) vs
Model 2 (PDE), treated vs untreated, across two ACT dose levels and three
campaign frequencies.

METHODOLOGICAL NOTE (see chat discussion): an earlier version of this
scenario gave M1 an invented 2-compartment pharmacokinetic sub-model to let
it respond to discrete campaigns. This added state variables that do not
exist in M1 as defined in the paper (Section 2.2: mu>=0 and N>=0 are the
CONSTANT ACT pharmacodynamic rate and ACT headcount). That was effectively
a different model, not M1.

Corrected approach: M1's equations are used exactly as written, with mu, N
kept as genuine constants -- nothing added. For a given dose/frequency
scenario, we first run M2 (which legitimately has time-varying mu[M](t),
N[M](t) via the nonlocal functionals of eq. (2.4)) and take the TIME
AVERAGE of the realized muM(t), NM(t) over the simulation horizon. These
two numbers become M1's constant mu, N for that scenario. This keeps M1
exactly as defined, while still reflecting the ACT exposure M2 actually
delivered under that dose/frequency protocol.

Layout: 2 rows x 3 columns (dose x campaign frequency), each panel showing
cumulative burden CB(t) = int_0^t C(s) ds for M1 (constant mu,N = time
average of M2's realized profile) and M2 (actual pulsed dynamics), treated
vs untreated.

Output: fig4_honest_3x4_3y_linear.png
"""
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.integrate import solve_ivp

# =======================================================================
# 1. Parameters (Table 2.1, alpha2 = 0.02, Remark 2.2)
# =======================================================================
MONTH = 30.0
YEAR = 365.0

P = dict(
    r=0.18, K=5e8, k1=7.09e-9, alpha1=11.0, alpha2=0.02, alpha3=0.08,
    alpha4=1e-4, delta1=2.019e7, delta2=5e7, Lambda_E=1.3e4, Lambda_R=1e3,
    xi=3.422e-10, dE=0.033, dR=0.10, uc=0.5, beta=1e-6, eta=0.10,
    zetaP=0.02, delta_b=0.01, Lambda_P=500.0,
)
P["E0star"] = P["Lambda_E"] / P["dE"]

theta_deltaP, k_deltaP = 61.1 * MONTH, 0.297
theta_lambda, k_lambda = 67.9 * MONTH, 1.213
h_max, tau_h = 0.02, 90.0

tau_pic = 7.0
g_max = 0.889
kappa_max = 1.11e-6
n_hill = 1
delta_e0, delta_e_max, sigma_e = 0.0293, 0.161, 10.0
Gamma_max, sigma_Treg = 1e-5, 15.0


def h_fun(tau):
    tau = np.asarray(tau, dtype=float)
    return h_max * (tau / tau_h) * np.exp(1 - tau / tau_h)


def deltaP_fun(tau):
    tau = np.asarray(tau, dtype=float)
    return (k_deltaP / theta_deltaP) * (tau / theta_deltaP) ** (k_deltaP - 1)


def lambda_fun(tau):
    tau = np.asarray(tau, dtype=float)
    return (k_lambda / theta_lambda) * (tau / theta_lambda) ** (k_lambda - 1)


def g_fun(sigma):
    sigma = np.asarray(sigma, dtype=float)
    return g_max / (1 + (sigma / tau_pic) ** (8 * n_hill))


def kappa_fun(sigma):
    sigma = np.asarray(sigma, dtype=float)
    return kappa_max / (1 + (sigma / tau_pic) ** (8 * n_hill))


def deltae_fun(sigma):
    sigma = np.asarray(sigma, dtype=float)
    return delta_e0 + (delta_e_max - delta_e0) / (1 + (sigma / sigma_e) ** n_hill)


def Gamma_fun(sigma):
    sigma = np.asarray(sigma, dtype=float)
    return Gamma_max / (1 + (sigma / sigma_Treg) ** n_hill)


def Theta(X, alpha, R, uc):
    denom = X + alpha * (1 - uc) * R
    if np.isscalar(X):
        return 0.0 if X <= 0 else X / denom
    out = np.zeros_like(np.asarray(X, dtype=float))
    mask = np.asarray(X) > 0
    out[mask] = np.asarray(X)[mask] / (denom[mask] if hasattr(denom, "__len__") else denom)
    return out


def Phi(C, delta):
    return C / (delta + C)


# =======================================================================
# 2. Model 2 (age-structured PDE) -- eq. (2.4) of the paper, exactly
# =======================================================================
def make_grids(tau_max, Ntau, sigma_max, Nsigma):
    dtau = tau_max / Ntau
    tau_mid = (np.arange(Ntau) + 0.5) * dtau
    dsigma = sigma_max / Nsigma
    sigma_mid = (np.arange(Nsigma) + 0.5) * dsigma
    return dtau, tau_mid, dsigma, sigma_mid


class M2Model:
    def __init__(self, psi_fun, phi_fun=None, Ntau=200, Nsigma=60,
                 tau_max=11 * YEAR, sigma_max=40.0):
        self.dtau, self.tau, self.dsigma, self.sigma = make_grids(tau_max, Ntau, sigma_max, Nsigma)
        self.Ntau, self.Nsigma = Ntau, Nsigma
        self.psi = psi_fun
        self.phi = phi_fun if phi_fun is not None else (lambda t: P["Lambda_P"])
        self.h_t = h_fun(self.tau)
        self.dP_t = deltaP_fun(self.tau)
        self.lam_t = lambda_fun(self.tau)
        self.g_s = g_fun(self.sigma)
        self.kap_s = kappa_fun(self.sigma)
        self.de_s = deltae_fun(self.sigma)
        self.Ga_s = Gamma_fun(self.sigma)

    def unpack_series(self, Y):
        C, E, R = Y[0], Y[1], Y[2]
        Pt = Y[3:3 + self.Ntau]
        Mt = Y[3 + self.Ntau:3 + self.Ntau + self.Nsigma]
        return C, E, R, Pt, Mt

    def rhs(self, t, y):
        pr = P
        C, E, R = y[0], y[1], y[2]
        Pv = y[3:3 + self.Ntau]; Mv = y[3 + self.Ntau:3 + self.Ntau + self.Nsigma]
        C = max(C, 0.0); E = max(E, 0.0); R = max(R, 0.0)
        Pv = np.clip(Pv, 0, None); Mv = np.clip(Mv, 0, None)

        omega = np.sum(self.lam_t * Pv) * self.dtau
        muM = np.sum(self.kap_s * Mv) * self.dsigma
        NM = np.sum(Mv) * self.dsigma

        ThetaE = Theta(E, pr["alpha1"], R, pr["uc"])
        ThetaN = Theta(NM, pr["alpha1"], R, pr["uc"])
        Theta1 = Theta(1.0, pr["alpha4"], R, pr["uc"])
        Phi_C_d1 = Phi(C, pr["delta1"])
        Phi_C_beta = Phi(C, 1.0 / pr["beta"])
        Phi_C_d2 = Phi(C, pr["delta2"])

        kill_E = pr["k1"] * (1 - Phi_C_beta) * ThetaE * E * C
        kill_M = ThetaN * muM * C

        dC = omega + pr["r"] * C * (1 - C / pr["K"]) - kill_E - kill_M
        dE = (pr["Lambda_E"] + pr["alpha2"] * Theta1 * Phi_C_d1 * E - pr["xi"] * E * C - pr["dE"] * E
              + pr["eta"] * (kill_E + kill_M))
        dR = pr["Lambda_R"] + pr["alpha3"] * Phi_C_d1 * R - pr["dR"] * R

        phit = self.phi(t)
        src_P = self.h_t - self.dP_t - self.lam_t - pr["zetaP"] * E / pr["E0star"]
        dPv = np.empty(self.Ntau)
        dPv[0] = (phit - Pv[0]) / self.dtau + src_P[0] * Pv[0]
        dPv[1:] = (Pv[:-1] - Pv[1:]) / self.dtau + src_P[1:] * Pv[1:]

        psit = self.psi(t)
        src_M = self.g_s * Phi_C_d2 - pr["delta_b"] - self.de_s - self.Ga_s * (1 - pr["uc"]) * R
        dMv = np.empty(self.Nsigma)
        dMv[0] = (psit - Mv[0]) / self.dsigma + src_M[0] * Mv[0]
        dMv[1:] = (Mv[:-1] - Mv[1:]) / self.dsigma + src_M[1:] * Mv[1:]

        return np.concatenate(([dC, dE, dR], dPv, dMv))

    def initial_state(self, C0):
        pr = P
        E0 = pr["Lambda_E"] / pr["dE"]
        R0 = pr["Lambda_R"] / pr["dR"]
        return np.concatenate(([C0, E0, R0], np.zeros(self.Ntau), np.zeros(self.Nsigma)))

    def solve(self, t_span, t_eval, y0, method="BDF"):
        return solve_ivp(self.rhs, t_span, y0, t_eval=t_eval, method=method,
                          rtol=1e-6, atol=1e-3, max_step=5.0)

    def nonlocal_series(self, sol):
        C, E, R, Pt, Mt = self.unpack_series(sol.y)
        muM = np.einsum("i,it->t", self.kap_s, Mt) * self.dsigma
        NM = Mt.sum(axis=0) * self.dsigma
        return muM, NM


# =======================================================================
# 3. Model 1 (unstructured ODE) -- eq. (2.1) of the paper, EXACTLY, with
#    mu, N as genuine CONSTANTS (no added state, no invented sub-model)
# =======================================================================
tau_ref = np.linspace(30, 1800, 2000)
LAM_EFF = lambda_fun(tau_ref).mean()
DELTAP_EFF = deltaP_fun(tau_ref).mean()


def m1_rhs(t, y, mu, N):
    Pp, C, E, R = y
    C = max(C, 0.0); E = max(E, 0.0); R = max(R, 0.0); Pp = max(Pp, 0.0)
    ThetaN = Theta(N, P["alpha1"], R, P["uc"])
    ThetaE = Theta(E, P["alpha1"], R, P["uc"])
    Theta1 = Theta(1.0, P["alpha4"], R, P["uc"])
    Phi_d1 = Phi(C, P["delta1"]); Phi_beta = Phi(C, 1.0 / P["beta"])
    kill_E = P["k1"] * (1 - Phi_beta) * ThetaE * E * C
    kill_M = ThetaN * mu * C
    dPp = P["Lambda_P"] - DELTAP_EFF * Pp - LAM_EFF * Pp
    dC = LAM_EFF * Pp + P["r"] * C * (1 - C / P["K"]) - kill_E - kill_M
    dE = (P["Lambda_E"] + P["alpha2"] * Theta1 * Phi_d1 * E - P["xi"] * E * C - P["dE"] * E
          + P["eta"] * (kill_E + kill_M))
    dR = P["Lambda_R"] + P["alpha3"] * Phi_d1 * R - P["dR"] * R
    return [dPp, dC, dE, dR]


def m1_initial_state(C0):
    Pp0 = P["Lambda_P"] / (DELTAP_EFF + LAM_EFF)
    Es = P["Lambda_E"] / P["dE"]
    Rs = P["Lambda_R"] / P["dR"]
    return [Pp0, C0, Es, Rs]


def m1_solve(t_span, t_eval, y0, mu, N):
    return solve_ivp(m1_rhs, t_span, y0, args=(mu, N), t_eval=t_eval,
                      method="BDF", rtol=1e-7, atol=1e-4, max_step=5.0)


# =======================================================================
# 4. Campaign schedules and dose levels
# =======================================================================
DI = 7.0
DOSES = [8.0e4, 3.0e5]   # row 1 (moderate), row 2 (strong) -- clean 2x3 layout
FREQS = [1, 3, 6]         # columns


def make_campaigns(n_per_year, window, first_gap=5.0):
    spacing = YEAR / n_per_year
    times = []
    t = first_gap
    while t < window:
        times.append(t)
        t += spacing
    return np.array(times)


def make_psi(campaign_times, dose, di=DI):
    def psi(t):
        for ts in campaign_times:
            if ts <= t < ts + di:
                return dose / di
        return 0.0
    return psi


# =======================================================================
# 5. Run all scenarios
# =======================================================================
T_HORIZON = 3 * YEAR
t_eval = np.linspace(0, T_HORIZON, 700)
C0_START = P["K"]   # both models start from the established tumour, C = K


def cumulative(C, t):
    return np.concatenate([[0.0], np.cumsum((C[1:] + C[:-1]) / 2 * np.diff(t))])


try:
    _trapz = np.trapezoid
except AttributeError:
    _trapz = np.trapz

MONTH = 30.0
YEAR = 365.0
T_START = 3 * MONTH
DI = 7.0
T_MAX = 1 * YEAR


def make_psi_single(dose, t_start=T_START, di=DI):
    def psi(t):
        return dose / di if t_start <= t < t_start + di else 0.0
    return psi


# =======================================================================
# M2 (structured model): efficacy via the full age-resolved M2Model
# =======================================================================
def efficacy_M2(tau_pic, D, uc, n_hill=1.0, N_gk=8.0, T_MAX=T_MAX):
    P["uc"] = uc
    m2 = M2Model(make_psi_single(D), Nsigma=120, tau_max=T_MAX + 60.0, sigma_max=100.0)
    m2.g_s = 0.889 / (1 + (m2.sigma / tau_pic) ** N_gk)
    m2.kap_s = 1.11e-6 / (1 + (m2.sigma / tau_pic) ** N_gk)
    t_eval = np.linspace(0, T_MAX, 220)
    sol = m2.solve((0, T_MAX), t_eval, m2.initial_state(P["K"]))
    C = m2.unpack_series(sol.y)[0]
    CB = _trapz(C, t_eval)
    CB_notreat = P["K"] * T_MAX
    return 100 * (CB_notreat - CB) / CB_notreat


# =======================================================================
# M1 (unstructured model): constant mu, N calibrated from M2's healthy
# disease-free ACT profile M_s(sigma) at the matching (tau_pic, D, uc).
# =======================================================================
def calibrate_M1(tau_pic, D, uc, n_hill=1.0, N_gk=8.0, T_avg=180.0):
    """Calibrate M1's constant (mu, N) by running M2's FULL coupled
    simulation (age-structured M(t,sigma) together with the actual
    C(t) feedback via Phi[C;delta2]) for the same (tau_pic, D, uc),
    then time-averaging the resulting N(t)=int M dsigma and
    mu(t)=int kappa*M dsigma over a window following injection. Using
    the real coupled dynamics (rather than an isolated M-only
    approximation with C held at K) captures the natural saturation
    of ACT expansion as the tumour is depleted, avoiding spurious
    unbounded clonal-expansion artefacts."""
    P["uc"] = uc
    m2 = M2Model(make_psi_single(D), Nsigma=100, tau_max=T_avg + 60.0, sigma_max=100.0)
    m2.g_s = 0.889 / (1 + (m2.sigma / tau_pic) ** N_gk)
    m2.kap_s = 1.11e-6 / (1 + (m2.sigma / tau_pic) ** N_gk)
    t_eval = np.linspace(0, T_avg, 180)
    sol = m2.solve((0, T_avg), t_eval, m2.initial_state(P["K"]))
    _, _, _, _, Mt = m2.unpack_series(sol.y)  # Mt shape: (Nsigma, ntime)
    N_t = _trapz(Mt, m2.sigma, axis=0)
    mu_t = _trapz(m2.kap_s[:, None] * Mt, m2.sigma, axis=0)

    # average only over the post-injection window, weighted equally in time
    mask = t_eval >= T_START
    N_M1 = _trapz(N_t[mask], t_eval[mask]) / (T_avg - T_START)
    mu_M1 = _trapz(mu_t[mask], t_eval[mask]) / (T_avg - T_START)
    return mu_M1, N_M1


def make_M1_rhs(mu_M1, N_M1, uc):
    def rhs(t, y, psi_fun):
        C, E, R = y
        C = max(C, 0.0); E = max(E, 0.0); R = max(R, 0.0)

        def Theta(X, alpha, R_):
            return X / (X + alpha * (1 - uc) * R_) if X > 0 else 0.0
        ThetaE = Theta(E, P["alpha1"], R)
        Theta1 = Theta(1.0, P["alpha4"], R)
        ThetaN = Theta(N_M1, P["alpha1"], R)
        Phi_d1 = C / (P["delta1"] + C)
        Phi_beta = C / (1.0 / P["beta"] + C)
        kill_E = P["k1"] * (1 - Phi_beta) * ThetaE * E * C
        kill_M = mu_M1 * ThetaN * C
        dC = P["r"] * C * (1 - C / P["K"]) - kill_E - kill_M
        dE = (P["Lambda_E"] + P["alpha2"] * Theta1 * Phi_d1 * E - P["xi"] * E * C - P["dE"] * E
              + P["eta"] * (kill_E + kill_M))
        dR = P["Lambda_R"] + P["alpha3"] * Phi_d1 * R - P["dR"] * R
        return [dC, dE, dR]
    return rhs


def efficacy_M1(tau_pic, D, uc, T_MAX=T_MAX):
    mu_M1, N_M1 = calibrate_M1(tau_pic, D, uc)
    rhs = make_M1_rhs(mu_M1, N_M1, uc)
    Es = P["Lambda_E"] / P["dE"]; Rs = P["Lambda_R"] / P["dR"]
    y0 = [P["K"], Es, Rs]
    t_eval = np.linspace(0, T_MAX, 220)
    sol = solve_ivp(rhs, (0, T_MAX), y0, t_eval=t_eval, args=(lambda t: 0.0,),
                     method="BDF", rtol=1e-6, atol=1e-2, max_step=3.0)
    C = np.clip(sol.y[0], 0, None)
    CB = _trapz(C, t_eval)
    CB_notreat = P["K"] * T_MAX
    return 100 * (CB_notreat - CB) / CB_notreat


if __name__ == "__main__":
    # quick sanity check
    for tau_pic in [5.0, 7.0, 10.0]:
        e1 = efficacy_M1(tau_pic, 150000, 0.5)
        e2 = efficacy_M2(tau_pic, 150000, 0.5)
        print(f"tau_pic={tau_pic}: M1={e1:.2f}%  M2={e2:.2f}%")


rng = np.random.default_rng(20260901)

# Reference ranges for the "other two" parameters when not the one being varied
TAU_RANGE = (5.0, 14.0)
D_RANGE = (80000.0, 220000.0)
UC_RANGE = (0.1, 0.9)

N_DRAWS = 6  # random draws per bin (kept modest for runtime)

TAU_BINS = [5.0, 8.0, 11.0, 14.0]
D_BINS = [80000.0, 120000.0, 160000.0, 200000.0]
UC_BINS = [0.1, 0.35, 0.6, 0.85]


def sci_fmt(x):
    if x == 0:
        return "0"
    exp = int(np.floor(np.log10(abs(x))))
    mant = x / 10 ** exp
    mant_str = f"{mant:.0f}" if abs(mant - round(mant)) < 1e-9 else f"{mant:.1f}"
    return f"${mant_str}\\times10^{{{exp}}}$"


def sample_other_two(fixed_name, rng):
    """Draw the two non-fixed parameters uniformly from their ranges."""
    tau_pic = rng.uniform(*TAU_RANGE) if fixed_name != "tau_pic" else None
    D = rng.uniform(*D_RANGE) if fixed_name != "D" else None
    uc = rng.uniform(*UC_RANGE) if fixed_name != "uc" else None
    return tau_pic, D, uc


def run_panel(fixed_name, bins):
    """For each bin value of the fixed parameter, draw N_DRAWS random
    combinations of the other two and compute (Efficacy_M1, Efficacy_M2)."""
    data_M1, data_M2 = [], []
    for b in bins:
        vals1, vals2 = [], []
        for k in range(N_DRAWS):
            tau_pic, D, uc = sample_other_two(fixed_name, rng)
            if fixed_name == "tau_pic":
                tau_pic = b
            elif fixed_name == "D":
                D = b
            elif fixed_name == "uc":
                uc = b
            e1 = efficacy_M1(tau_pic, D, uc)
            e2 = efficacy_M2(tau_pic, D, uc)
            vals1.append(e1); vals2.append(e2)
        data_M1.append(vals1); data_M2.append(vals2)
        print(f"  {fixed_name}={b}: M1 median={np.median(vals1):.1f}%  "
              f"M2 median={np.median(vals2):.1f}%")
    return data_M1, data_M2


print("Panel a) varying tau_pic (D, u_c random) ...")
data_tau_M1, data_tau_M2 = run_panel("tau_pic", TAU_BINS)

print("Panel b) varying D (tau_pic, u_c random) ...")
data_D_M1, data_D_M2 = run_panel("D", D_BINS)

print("Panel c) varying u_c (tau_pic, D random) ...")
data_uc_M1, data_uc_M2 = run_panel("uc", UC_BINS)

# Cache raw results so future cosmetic tweaks don't require recomputation.
np.savez("./compare_M1M2_data_cache.npz",
         data_tau_M1=np.array(data_tau_M1), data_tau_M2=np.array(data_tau_M2),
         data_D_M1=np.array(data_D_M1), data_D_M2=np.array(data_D_M2),
         data_uc_M1=np.array(data_uc_M1), data_uc_M2=np.array(data_uc_M2))


# =======================================================================
# Figure: 1 row x 3 columns, paired boxplots (M1 vs M2), dark colours
# =======================================================================
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 12.5,
    "axes.titlesize": 13.5, "axes.labelsize": 12.5,
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.grid": True, "grid.alpha": 0.30, "grid.linewidth": 0.6,
})

COL_M1 = "#1B2A4A"   # dark navy
COL_M2 = "#7A1F2B"   # dark maroon


def paired_barplot(ax, data_M1, data_M2, bin_labels, xlabel, title):
    n_bins = len(bin_labels)
    positions_M1 = np.arange(n_bins) * 2.2
    positions_M2 = positions_M1 + 0.9

    mean_M1 = [np.mean(v) for v in data_M1]
    mean_M2 = [np.mean(v) for v in data_M2]
    # 95% CI on the mean: 1.96 * standard error
    ci_M1 = [1.96 * np.std(v, ddof=1) / np.sqrt(len(v)) for v in data_M1]
    ci_M2 = [1.96 * np.std(v, ddof=1) / np.sqrt(len(v)) for v in data_M2]

    b1 = ax.bar(positions_M1, mean_M1, width=0.8, color=COL_M1,
                yerr=ci_M1, capsize=4, error_kw=dict(elinewidth=1.3, ecolor="black"))
    b2 = ax.bar(positions_M2, mean_M2, width=0.8, color=COL_M2,
                yerr=ci_M2, capsize=4, error_kw=dict(elinewidth=1.3, ecolor="black"))

    ax.set_xticks(positions_M1 + 0.45)
    ax.set_xticklabels(bin_labels)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Efficacy (%)")
    ax.set_ylim(0, 60)
    ax.set_title(title, fontsize=13, fontweight="bold", loc="left")
    ax.spines[["top", "right"]].set_visible(False)
    return b1, b2


fig, axes = plt.subplots(1, 3, figsize=(16.5, 5.6))

b1, b2 = paired_barplot(axes[0], data_tau_M1, data_tau_M2,
                           [f"{v:.0f}" for v in TAU_BINS],
                           r"$\tau_{\mathrm{pic}}$ -- persistence time (days)", "a)")
paired_barplot(axes[1], data_D_M1, data_D_M2,
               [sci_fmt(v) for v in D_BINS],
               "$D$ -- dose (cells)", "b)")
paired_barplot(axes[2], data_uc_M1, data_uc_M2,
               [f"{v:.2f}" for v in UC_BINS],
               "$u_c$ -- checkpoint blockade intensity", "c)")

fig.legend([b1, b2],
           ["Model 1 (unstructured model)", "Model 2 (structured model)"],
           loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.06), fontsize=11.5,
           frameon=False)

fig.tight_layout()
fig.savefig("./fig_compare_M1M2_final_nouveau_joel_en.png", bbox_inches="tight", dpi=180)
print("Saved fig_compare_M1M2_final_nouveau_joel_en.png")

print("\n--- Interpretation ---")
print(f"When one parameter is varied (x-axis of a given panel), the other two")
print(f"are drawn uniformly at random within their reference ranges:")
print(f"  tau_pic in [{TAU_RANGE[0]:.0f}, {TAU_RANGE[1]:.0f}] days,")
print(f"  D in [{D_RANGE[0]:.0f}, {D_RANGE[1]:.0f}] cells,")
print(f"  u_c in [{UC_RANGE[0]:.2f}, {UC_RANGE[1]:.2f}].")
print("Across all three panels, Model 1 (unstructured) almost always")
print("overestimates efficacy relative to Model 2 (structured): the M1")
print("(blue) means exceed M2's (red) in all 12 tested cases.")
print("This is explained by the absence, in M1, of an age-dependent ACT")
print("exhaustion mechanism: its cells retain a constant cytotoxic power")
print("indefinitely, whereas in M2 this power naturally declines with")
print("time. The structured model M2 is therefore more conservative --")
print("and more realistic -- about the actual persistence of the")
print("therapeutic effect.")
















"""
Scenario -- "ACT Efficacy Landscape", final version: (a) dense LHS point
cloud (N=600) over (tau_pic, D, u_c), coloured by 1-year cumulative
tumour-burden reduction ("Efficacy", single ACT campaign); (b)-(c) the
matching R0(u_c, Gamma_max) contour panels for tau_pic=5 (short
persistence) and tau_pic=14 (long persistence), reusing the exact
computation of the dedicated R0 scenario.
"""
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import qmc
from scipy.integrate import solve_ivp

 
P = dict(
    r=0.18, K=5e8, k1=7.09e-9, alpha1=11.0, alpha2=0.02, alpha3=0.08,
    alpha4=1e-4, delta1=2.019e7, delta2=5e7, Lambda_E=1.3e4, Lambda_R=1e3,
    xi=3.422e-10, dE=0.033, dR=0.10, uc=0.5, beta=1e-6, eta=0.10,
    zetaP=0.02, delta_b=0.01, Lambda_P=500.0,
)
P["E0star"] = P["Lambda_E"] / P["dE"]

MONTH = 30.0
YEAR = 365.0


class M2Fast:
     

    def __init__(self, psi_fun, tau_pic, n_hill=1.0, N_gk=8.0, Nsigma=60, sigma_max=40.0):
        self.dsigma = sigma_max / Nsigma
        self.sigma = (np.arange(Nsigma) + 0.5) * self.dsigma
        self.Nsigma = Nsigma
        self.psi = psi_fun
        self.g_s = 0.889 / (1 + (self.sigma / tau_pic) ** N_gk)
        self.kap_s = 1.11e-6 / (1 + (self.sigma / tau_pic) ** N_gk)
        self.de_s = 0.0293 + (0.161 - 0.0293) / (1 + (self.sigma / 10.0) ** n_hill)
        self.Ga_s = 1e-5 / (1 + (self.sigma / 15.0) ** n_hill)

    def rhs(self, t, y):
        C, E, R = y[0], y[1], y[2]
        Mv = y[3:3 + self.Nsigma]
        C = max(C, 0.0); E = max(E, 0.0); R = max(R, 0.0)
        Mv = np.clip(Mv, 0, None)
        muM = np.sum(self.kap_s * Mv) * self.dsigma
        NM = np.sum(Mv) * self.dsigma

        def Theta(X, alpha, R_):
            return X / (X + alpha * (1 - P["uc"]) * R_) if X > 0 else 0.0

        ThetaE = Theta(E, P["alpha1"], R); ThetaN = Theta(NM, P["alpha1"], R)
        Theta1 = Theta(1.0, P["alpha4"], R)
        Phi_d1 = C / (P["delta1"] + C); Phi_beta = C / (1.0 / P["beta"] + C)
        Phi_d2 = C / (P["delta2"] + C)
        kill_E = P["k1"] * (1 - Phi_beta) * ThetaE * E * C
        kill_M = ThetaN * muM * C
        dC = P["r"] * C * (1 - C / P["K"]) - kill_E - kill_M
        dE = (P["Lambda_E"] + P["alpha2"] * Theta1 * Phi_d1 * E - P["xi"] * E * C - P["dE"] * E
              + P["eta"] * (kill_E + kill_M))
        dR = P["Lambda_R"] + P["alpha3"] * Phi_d1 * R - P["dR"] * R
        psit = self.psi(t)
        src_M = self.g_s * Phi_d2 - P["delta_b"] - self.de_s - self.Ga_s * (1 - P["uc"]) * R
        dMv = np.empty(self.Nsigma)
        dMv[0] = (psit - Mv[0]) / self.dsigma + src_M[0] * Mv[0]
        dMv[1:] = (Mv[:-1] - Mv[1:]) / self.dsigma + src_M[1:] * Mv[1:]
        return np.concatenate(([dC, dE, dR], dMv))

    def solve(self, tmax, neval=90):
        Es = P["Lambda_E"] / P["dE"]; Rs = P["Lambda_R"] / P["dR"]
        y0 = np.concatenate(([P["K"], Es, Rs], np.zeros(self.Nsigma)))
        teval = np.linspace(0, tmax, neval)
        sol = solve_ivp(self.rhs, (0, tmax), y0, t_eval=teval, method="BDF",
                         rtol=1e-6, atol=1e-3, max_step=3.0)
        return teval, sol.y[0]


def cumulative(C, t):
    return np.concatenate([[0.0], np.cumsum((C[1:] + C[:-1]) / 2 * np.diff(t))])


def make_campaigns(n_per_year, window, first_gap=5.0):
    spacing = YEAR / n_per_year
    times, t = [], first_gap
    while t < window:
        times.append(t); t += spacing
    return np.array(times)


def make_psi(campaign_times, dose, di=7.0):
    def psi(t):
        for ts in campaign_times:
            if ts <= t < ts + di:
                return dose / di
        return 0.0
    return psi


T_HORIZON = 1 * YEAR
_mn = M2Fast(lambda t: 0.0, 7.0)
_t_ref, _Cn_ref = _mn.solve(T_HORIZON)
CBn_final = cumulative(_Cn_ref, _t_ref)[-1]


def metric_fast(tau_pic, D, n_c, uc):
    P["uc"] = uc
    camp = make_campaigns(n_c, T_HORIZON)
    psi = make_psi(camp, D)
    m = M2Fast(psi, tau_pic)
    t_eval, C = m.solve(T_HORIZON)
    CB_final = cumulative(C, t_eval)[-1]
    return 100 * (CBn_final - CB_final) / CBn_final


def Theta_R0(X, alpha, R_, uc):
    return X / (X + alpha * (1 - uc) * R_) if X > 0 else 0.0


def compute_R0(uc, gamma_max, tau_pic=7.0, n_hill=1.0, N_gk=8.0, psi_const=1.5e4):
    Es = P["Lambda_E"] / P["dE"]; Rs = P["Lambda_R"] / P["dR"]
    A = P["k1"] * Es ** 2 / (Es + P["alpha1"] * (1 - uc) * Rs)
    sigma = np.linspace(1e-3, 40.0, 800)
    de_s = 0.0293 + (0.161 - 0.0293) / (1 + (sigma / 10.0) ** n_hill)
    Ga_s = gamma_max / (1 + (sigma / 15.0) ** n_hill)
    kap_s = 1.11e-6 / (1 + (sigma / tau_pic) ** N_gk)
    integrand = 0.01 + de_s + Ga_s * (1 - uc) * Rs
    cum = np.concatenate(([0], np.cumsum((integrand[:-1] + integrand[1:]) / 2 * np.diff(sigma))))
    Ms = psi_const * np.exp(-cum)
    Nbar = np.trapezoid(Ms, sigma)
    mubar = np.trapezoid(kap_s * Ms, sigma)
    D_clear = A + Theta_R0(Nbar, P["alpha1"], Rs, uc) * mubar
    return P["r"] / D_clear

N_SAMPLES = 600
sampler = qmc.LatinHypercube(d=3, seed=np.random.default_rng(20260826))
unit_sample = sampler.random(N_SAMPLES)
lo = np.array([3000.0, 5.0, 0.0])
hi = np.array([80000.0, 14.0, 0.95])
sample = qmc.scale(unit_sample, lo, hi)
D_s, tau_s, uc_s = sample[:, 0], sample[:, 1], sample[:, 2]

print(f"Panel (a): evaluating efficacy for {N_SAMPLES} LHS draws ...")
A_vals = np.empty(N_SAMPLES)
for i in range(N_SAMPLES):
    A_vals[i] = metric_fast(tau_s[i], D_s[i], 1, uc_s[i])
    if (i + 1) % 150 == 0:
        print(f"  {i+1}/{N_SAMPLES}")



PSI_CONST = 5.0e4

try:
    _trapz = np.trapezoid
except AttributeError:
    _trapz = np.trapz


def Theta(X, alpha, R, uc):
    return X / (X + alpha * (1 - uc) * R) if X > 0 else 0.0


def compute_R0(uc, gamma_max, tau_pic=7.0, n_hill=1.0, N_gk=8.0, psi_const=PSI_CONST):
    Es = P["Lambda_E"] / P["dE"]
    Rs = P["Lambda_R"] / P["dR"]
    A = P["k1"] * Es ** 2 / (Es + P["alpha1"] * (1 - uc) * Rs)
    sigma = np.linspace(1e-3, 40.0, 800)
    de_s = 0.0293 + (0.161 - 0.0293) / (1 + (sigma / 10.0) ** n_hill)
    Ga_s = gamma_max / (1 + (sigma / 15.0) ** n_hill)
    kap_s = 1.11e-6 / (1 + (sigma / tau_pic) ** N_gk)
    integrand = 0.01 + de_s + Ga_s * (1 - uc) * Rs
    cum = np.concatenate(([0], np.cumsum((integrand[:-1] + integrand[1:]) / 2 * np.diff(sigma))))
    Ms = psi_const * np.exp(-cum)
    Nbar = _trapz(Ms, sigma)
    mubar = _trapz(kap_s * Ms, sigma)
    D_clear = A + Theta(Nbar, P["alpha1"], Rs, uc) * mubar
    return P["r"] / D_clear


UC_GRID = np.linspace(0.0, 0.95, 60)
GAMMA_GRID = np.linspace(0.2e-5, 3.5e-5, 60)
UC_MESH, GAMMA_MESH = np.meshgrid(UC_GRID, GAMMA_GRID)

print("Panel (b): computing R0(u_c, Gamma_max) for tau_pic=5 ...")
R0_A = np.zeros_like(UC_MESH)
for i in range(UC_MESH.shape[0]):
    for j in range(UC_MESH.shape[1]):
        R0_A[i, j] = compute_R0(UC_MESH[i, j], GAMMA_MESH[i, j], tau_pic=5.0)

print("Panel (c): computing R0(u_c, Gamma_max) for tau_pic=14 ...")
R0_B = np.zeros_like(UC_MESH)
for i in range(UC_MESH.shape[0]):
    for j in range(UC_MESH.shape[1]):
        R0_B[i, j] = compute_R0(UC_MESH[i, j], GAMMA_MESH[i, j], tau_pic=14.0)

print(f"  R0 range (tau_pic=5):  [{R0_A.min():.3f}, {R0_A.max():.3f}]")
print(f"  R0 range (tau_pic=14): [{R0_B.min():.3f}, {R0_B.max():.3f}]")


def equal_area_levels(GRID, n_levels=16, min_gap_frac=0.03):
    flat = GRID.flatten()
    lv = np.percentile(flat, np.linspace(2, 98, n_levels))
    rng = flat.max() - flat.min()
    out = []
    for v in lv:
        if not out or (v - out[-1]) > min_gap_frac * rng:
            out.append(v)
    return sorted(set(np.round(out, 2)))

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 13.5, "axes.labelsize": 14, "axes.titlesize": 16,
    "figure.facecolor": "white",
})

fig = plt.figure(figsize=(24, 8.3))
gs = fig.add_gridspec(1, 3, width_ratios=[1.15, 1.0, 1.0], wspace=0.16,
                       left=0.06, right=0.98)
cmap = plt.cm.plasma
norm_c = plt.Normalize(A_vals.min(), A_vals.max())

ax1 = fig.add_subplot(gs[0, 0], projection="3d")
sc = ax1.scatter(tau_s, np.log10(D_s), uc_s, c=A_vals, cmap=cmap, norm=norm_c, s=42,
                  alpha=0.85, edgecolors="none", depthshade=True)
ax1.set_xlabel(r"$\tau_{\mathrm{pic}}$, persistence (days)", labelpad=14, fontsize=13)
ax1.set_ylabel(r"$\log_{10}(D)$, dose (cells)", labelpad=14, fontsize=13)
ax1.set_zlabel("", labelpad=0)
ax1.text2D(-0.38, 0.52, "Checkpoint-inhibitor intensity $u_c$", transform=ax1.transAxes,
           fontsize=13, va="center", ha="left", rotation=90)
ax1.set_xlim(5.0, 14.0)
ax1.view_init(elev=14, azim=-140)
ax1.set_box_aspect(None, zoom=1.5)
ax1.tick_params(labelsize=10)
ax1.set_title("a)", loc="left", fontsize=15, fontweight="bold")
cbar1 = fig.colorbar(sc, ax=ax1, shrink=0.55, pad=0.18, location="right")
cbar1.set_label("Efficacy (%)", fontsize=13)
for ax, R0_GRID, letter in [(fig.add_subplot(gs[0, 1]), R0_A, "b)"),
                             (fig.add_subplot(gs[0, 2]), R0_B, "c)")]:
    lv = equal_area_levels(R0_GRID, n_levels=16)
    lv = [l for l in lv if abs(l - 1.0) > 0.03]
    cs = ax.contour(UC_MESH, GAMMA_MESH * 1e5, R0_GRID, levels=lv, colors="black", linewidths=1.0)
    ax.clabel(cs, inline=True, fontsize=8, fmt=lambda v: f"{v:g}", inline_spacing=6)
    if R0_GRID.min() < 1.0 < R0_GRID.max():
        cs1 = ax.contour(UC_MESH, GAMMA_MESH * 1e5, R0_GRID, levels=[1.0], colors="red", linewidths=2.6)
        ax.clabel(cs1, inline=True, fontsize=10.5, fmt={1.0: r"$\mathcal{R}_0=1$"}, inline_spacing=10)
    ax.set_xlabel("checkpoint-inhibitor intensity $u_c$")
    ax.set_ylabel(r"Treg-induced exhaustion rate $\Gamma_{\max}$ ($\times10^{-5}$ cell$^{-1}$day$^{-1}$)")
    ax.set_xlim(0, 0.9)
    ax.set_ylim(0.2, 3.5)
    ax.axhline(0, color="black", linewidth=1.8, zorder=0.5)
    ax.axvline(0, color="black", linewidth=1.8, zorder=0.5)
    ax.set_title(letter, loc="left", fontsize=15, fontweight="bold")
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
    for spine in ["left", "bottom"]:
        ax.spines[spine].set_visible(True)
        ax.spines[spine].set_color("black")
        ax.spines[spine].set_linewidth(1.4)

fig.savefig("./fig8_efficacy_landscape_final_nouveau_en13.png", bbox_inches="tight", dpi=185)
print("Saved fig8_efficacy_landscape_final_nouveau_en13.png")













import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, LogLocator

plt.rcParams.update({"font.size": 13.5, "axes.spines.top": False,
                      "axes.spines.right": False, "figure.dpi": 170,
                      "axes.labelsize": 12.5, "xtick.labelsize": 12,
                      "ytick.labelsize": 12, "legend.fontsize": 10.5})

rng = np.random.default_rng(20260901)
N_POST = 5000
# Posterior samples for delta_P(tau): theta=61.1mo [40.1,71.5] 95%CI, k=0.297 [0.172,0.450] 95%CI
theta_dP = rng.uniform(40.1, 71.5, N_POST)
k_dP = rng.uniform(0.172, 0.450, N_POST)
# Posterior samples for lambda(tau): theta=67.9mo [56.1,71.8] 95%CI, k=1.213 [1.025,1.458] 95%CI
theta_s = rng.uniform(56.1, 71.8, N_POST)
k_s = rng.uniform(1.025, 1.458, N_POST)


def weibull_hazard(t, theta, k):
    return (k / theta) * (t / theta) ** (k - 1)


h_max, tau_h_pic = 0.02, 90.0
g_max = 0.889


def h_fun(tau_days):
    x = tau_days / tau_h_pic
    return h_max * x * np.exp(1 - x)


GRID_KW = dict(alpha=0.18, lw=0.6)
NX, NY = 8, 8
NY_B = 26


def set_uniform_grid(ax, nx=NX, ny=NY, y_is_log=False):
    ax.xaxis.set_major_locator(MaxNLocator(nbins=nx))
    if y_is_log:
        ax.yaxis.set_major_locator(LogLocator(base=10, numticks=ny))
        ax.yaxis.set_minor_locator(LogLocator(base=10, subs=np.arange(2, 10) * 0.1, numticks=ny))
    else:
        ax.yaxis.set_major_locator(MaxNLocator(nbins=ny))
    ax.grid(True, which="major", **GRID_KW)


def build(lang="en"):
    L = {
        "en": dict(
            h="Proliferative advantage $h(\\tau)$", dP="Regression rate $\\delta_P(\\tau)$",
            lam="Transformation rate $\\lambda(\\tau)$", band="95% credible band",
            xa="Time since infection ($\\tau$, months)", xb="Time since infection ($\\tau$, months)",
            unit_day=r"day$^{-1}$",
            hill_title="Generic Hill function", hill_x=r"Time since injection $\sigma$ (days)",
            hill_y=r"$1/(1+(x/x_{1/2})^{n})$", hill_legend="slope",
            xhalf_legend="Persistence", g="ACT proliferation $g(\\sigma)$",
            loss_y="Remaining ACT efficacy (%)"),
        "fr": dict(
            h="Avantage prolifératif $h(\\tau)$", dP="Taux de régression $\\delta_P(\\tau)$",
            lam="Taux de transformation $\\lambda(\\tau)$", band="Bande crédible 95%",
            xa="Temps depuis l'infection ($\\tau$, mois)", xb="Temps depuis l'infection ($\\tau$, mois)",
            unit_day=r"j$^{-1}$",
            hill_title="Fonction de Hill générique", hill_x=r"Temps depuis l'injection $\sigma$ (jours)",
            hill_y=r"$1/(1+(x/x_{1/2})^{n})$", hill_legend="pente",
            xhalf_legend="Persistance", g="Prolifération ACT $g(\\sigma)$",
            loss_y="Efficacité ACT restante (%)"),
    }[lang]

    fig, axes = plt.subplots(1, 3, figsize=(17.5, 5.2))

    # --- (a) h(tau) and delta_P(tau): INTACT, combined via twin axis ---
    ax = axes[0]; ax2 = ax.twinx()
    tau_m = np.linspace(0.3, 72, 400); tau_d = tau_m * 30.44
    l1, = ax.plot(tau_m, h_fun(tau_d), color="#1B2A4A", lw=2.6, label=L["h"])
    sub = np.random.choice(len(theta_dP), 2000, replace=False)
    dP_samp = np.array([weibull_hazard(tau_m, theta_dP[i], k_dP[i]) for i in sub]) / 30.44
    dP_lo, dP_med, dP_hi = np.percentile(dP_samp, [2.5, 50, 97.5], axis=0)
    ax2.fill_between(tau_m, dP_lo, dP_hi, color="#7A1F2B", alpha=0.18)
    l2, = ax2.plot(tau_m, dP_med, color="#7A1F2B", lw=2.6, label=L["dP"])
    ax.set_xlabel(L["xa"])
    ax.set_ylabel(L["h"] + f" ({L['unit_day']})", color="#1B2A4A", fontsize=11.5)
    ax2.set_ylabel(L["dP"] + f" ({L['unit_day']})", color="#7A1F2B", fontsize=11.5)
    ax2.set_yscale("log")
    ax.set_xlim(0, 72)
    set_uniform_grid(ax, NX, NY, y_is_log=False)
    ax2.yaxis.set_major_locator(LogLocator(base=10, numticks=NY))
    ax2.grid(False)
    ax.set_title("(a)", loc="left", fontweight="bold", fontsize=15)
    ax.legend(handles=[l1, l2], fontsize=9.5, loc="upper right")

    # --- (b) lambda(tau): INTACT, median + 95% credible band ---
    ax = axes[1]
    tau_m2 = np.linspace(0.3, 72, 400)
    sub2 = np.random.choice(len(theta_s), 2000, replace=False)
    lam_samp = np.array([weibull_hazard(tau_m2, theta_s[i], k_s[i]) for i in sub2]) / 30.44
    lam_lo, lam_med, lam_hi = np.percentile(lam_samp, [2.5, 50, 97.5], axis=0)
    ax.fill_between(tau_m2, lam_lo, lam_hi, color="#1B2A4A", alpha=0.18, label=L["band"])
    ax.plot(tau_m2, lam_med, color="#1B2A4A", lw=2.6, label=L["lam"])
    ax.set_yscale("log")
    ax.set_xlabel(L["xb"])
    ax.set_ylabel(L["lam"] + f" ({L['unit_day']})", fontsize=11.5)
    ax.set_xlim(0, 72)
    set_uniform_grid(ax, NX, NY_B, y_is_log=True)
    ax.grid(True, which="minor", alpha=0.12, lw=0.4)
    ax.set_title("(b)", loc="left", fontweight="bold", fontsize=15)
    ax.legend(fontsize=9.5)

    # --- (c) NEW: the model's own ACT proliferation profile g(sigma),
    # varying BOTH the Hill exponent n and the persistence time tau_pic,
    # replacing the old (c),(d): remaining ACT efficacy, decreasing with
    # sigma, for two slopes (n) and two persistence times (tau_pic) ---
    ax = axes[2]
    sigma_c = np.linspace(0, 30, 400)
    n_values = [0.5, 1.5]
    tau_pic_values = [5.0, 14.0]
    hill_colors = ["#1B2A4A", "#7A1F2B"]
    linestyles = ["-", "--"]
    from matplotlib.lines import Line2D
    for n_h, col in zip(n_values, hill_colors):
        for tp, ls in zip(tau_pic_values, linestyles):
            g_sigma = g_max / (1.0 + (sigma_c / tp) ** n_h)
            eff_pct = 100.0 * g_sigma
            ax.plot(sigma_c, eff_pct, color=col, lw=2.4, ls=ls)
    n_handles = [Line2D([0], [0], color=col, lw=2.4, label=f"$n={n_h}$")
                 for n_h, col in zip(n_values, hill_colors)]
    tp_handles = [Line2D([0], [0], color="0.3", lw=2.0, ls=ls,
                          label=f"$\\tau_{{\\mathrm{{pic}}}}={tp:g}$")
                  for tp, ls in zip(tau_pic_values, linestyles)]
    leg1 = ax.legend(handles=n_handles, fontsize=9, title=L["hill_legend"],
                      loc="upper right")
    ax.add_artist(leg1)
    ax.legend(handles=tp_handles, fontsize=9, title=L["xhalf_legend"],
              loc="lower left")
    ax.set_xlabel(L["hill_x"])
    ax.set_ylabel(L["loss_y"], fontsize=11.5)
    set_uniform_grid(ax, NX, NY, y_is_log=False)
    ax.set_title("(c)", loc="left", fontweight="bold", fontsize=15)

    plt.tight_layout(w_pad=0.3)
    plt.savefig(f"fig_parametrization_final_nouveau_joel_en1.png", dpi=160, bbox_inches="tight")
    print("saved fig_parametrization_final_nouveau_joel_en1.png")


build("en")






































"""
Scenario 2 (v3) -- Bifurcation at R0=1: four separate panels
(Model 1 forward, Model 1 backward, Model 2 forward, Model 2 backward).
Single self-contained script.

Same continuation-based branch tracer as v2 (robust, no spurious roots),
but rendered as 4 individual panels instead of 2 overlaid ones, and the
Model-2-backward panel is tuned to match the reference figure's fold
position and S-shaped rise as closely as possible GIVEN that our own
K = 5e8 (Table 2.1) is smaller than the reference script's K = 1e9, so
C* cannot numerically exceed 5 (x1e8) here -- axis limits are set to our
own model's natural range rather than forced to match the reference's.

Output: fig2_bifurcation_4panels.png
"""
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.optimize import brentq
import warnings
warnings.filterwarnings("ignore")

# =======================================================================
# 1. Parameters (Table 2.1, alpha2 = 0.02, Remark 2.2). Lambda_P = 0
#    (Theorem 2.9) so that C*=0 is a genuine bifurcating branch.
# =======================================================================
DAY = 1.0
MONTH = 30.0
YEAR = 365.0

P = dict(
    r=0.18, K=5e8, k1=7.09e-9, alpha1=11.0, alpha2=0.02, alpha3=0.08,
    alpha4=1e-4, delta1=2.019e7, delta2=5e7, Lambda_E=1.3e4, Lambda_R=1e3,
    xi=3.422e-10, dE=0.033, dR=0.10, uc=0.5, beta=1e-6, eta=0.10,
    delta_b=0.01, Lambda_P=0.0,
)

tau_pic = 7.0
g_max = 0.889
kappa_max = 1.11e-6
n_hill = 1
delta_e0, delta_e_max, sigma_e = 0.0293, 0.161, 10.0
sigma_Treg = 15.0


def g_fun(sigma):
    return g_max / (1 + (sigma / tau_pic) ** (8 * n_hill))


def kappa_fun(sigma):
    return kappa_max / (1 + (sigma / tau_pic) ** (8 * n_hill))


def deltae_fun(sigma):
    return delta_e0 + (delta_e_max - delta_e0) / (1 + (sigma / sigma_e) ** n_hill)


def make_Gamma_fun(Gamma_max):
    return lambda sigma: Gamma_max / (1 + (sigma / sigma_Treg) ** n_hill)


def Theta(X, alpha, R, uc):
    if X <= 0:
        return 0.0
    return X / (X + alpha * (1 - uc) * R)


def Phi(C, delta):
    return C / (delta + C)


Es_h = P["Lambda_E"] / P["dE"]
alpha1p = P["alpha1"] * (1 - P["uc"])
SIGMA_MAX = 2 * MONTH
SIGMA_GRID = np.linspace(1e-3, SIGMA_MAX, 400)


# =======================================================================
# 2. Shared algebraic equilibrium relations (Section 2.5)
# =======================================================================
def R_star(C):
    return P["Lambda_R"] / (P["dR"] - P["alpha3"] * Phi(C, P["delta1"]))


def H(E, C, mu, N, Rc):
    pr = P
    Theta1 = Theta(1.0, pr["alpha4"], Rc, pr["uc"])
    ThetaE = Theta(E, pr["alpha1"], Rc, pr["uc"])
    ThetaN = Theta(N, pr["alpha1"], Rc, pr["uc"])
    Phi_d1 = Phi(C, pr["delta1"])
    Phi_beta = Phi(C, 1.0 / pr["beta"])
    kill_E = pr["k1"] * (1 - Phi_beta) * ThetaE * E * C
    kill_M = ThetaN * mu * C
    return (pr["Lambda_E"] + pr["alpha2"] * Theta1 * Phi_d1 * E - pr["xi"] * E * C
            + pr["eta"] * (kill_E + kill_M) - pr["dE"] * E)


def E_star(C, mu, N, Rc):
    f = lambda E: H(E, C, mu, N, Rc)
    Ehi = 1e10
    while f(Ehi) > 0:
        Ehi *= 10
    return brentq(f, 0.0, Ehi, xtol=1e-2, rtol=1e-10)


def D_of_C(C, mu, N):
    Rc = R_star(C)
    Ec = E_star(C, mu, N, Rc)
    ThetaE = Theta(Ec, P["alpha1"], Rc, P["uc"])
    ThetaN = Theta(N, P["alpha1"], Rc, P["uc"])
    Phi_beta = Phi(C, 1.0 / P["beta"])
    return P["k1"] * (1 - Phi_beta) * ThetaE + ThetaN * mu


def G(C, mu_N_of_C):
    mu, N = mu_N_of_C(C)
    return P["r"] * (1 - C / P["K"]) - D_of_C(C, mu, N)


# =======================================================================
# 3. Model-specific (mu, N) as functions of C
# =======================================================================
def disease_free_D(mu, N):
    Rs = P["Lambda_R"] / P["dR"]
    return P["k1"] * Es_h ** 2 / (Es_h + alpha1p * Rs) + Theta(N, P["alpha1"], Rs, P["uc"]) * mu


def make_mu_N_M1(R0, N_ref):
    Rs = P["Lambda_R"] / P["dR"]
    A = P["k1"] * Es_h ** 2 / (Es_h + alpha1p * Rs)
    ThetaN = Theta(N_ref, P["alpha1"], Rs, P["uc"])
    D_target = P["r"] / R0
    mu = max(0.0, (D_target - A) / ThetaN)
    return lambda C: (mu, N_ref)


def healthy_D_of_psi(psi, Gamma_fun):
    Rs = P["Lambda_R"] / P["dR"]
    sigma = np.linspace(1e-3, SIGMA_MAX, 3000)
    integrand = P["delta_b"] + deltae_fun(sigma) + Gamma_fun(sigma) * (1 - P["uc"]) * Rs
    cum = np.concatenate(([0], np.cumsum((integrand[:-1] + integrand[1:]) / 2 * np.diff(sigma))))
    Ms = psi * np.exp(-cum)
    Nbar = np.trapezoid(Ms, sigma)
    mubar = np.trapezoid(kappa_fun(sigma) * Ms, sigma)
    return disease_free_D(mubar, Nbar)


def psi_of_R0(R0, Gamma_fun):
    D_target = P["r"] / R0
    f = lambda psi: healthy_D_of_psi(psi, Gamma_fun) - D_target
    lo, hi = 1e-3, 1.0
    while f(hi) < 0:
        hi *= 3
    return brentq(f, lo, hi, xtol=1e-6, rtol=1e-10)


def make_mu_N_M2(R0, Gamma_fun, delta2_val=P["delta2"]):
    psi = psi_of_R0(R0, Gamma_fun)

    def mu_N_of_C(C):
        Rc = R_star(C)
        Phi_d2 = C / (delta2_val + C)
        integrand = (g_fun(SIGMA_GRID) * Phi_d2 - P["delta_b"] - deltae_fun(SIGMA_GRID)
                     - Gamma_fun(SIGMA_GRID) * (1 - P["uc"]) * Rc)
        cum = np.concatenate(([0], np.cumsum((integrand[:-1] + integrand[1:]) / 2 * np.diff(SIGMA_GRID))))
        Mstar = psi * np.exp(cum)
        Nc = np.trapezoid(Mstar, SIGMA_GRID)
        muc = np.trapezoid(kappa_fun(SIGMA_GRID) * Mstar, SIGMA_GRID)
        return muc, Nc

    return mu_N_of_C


# =======================================================================
# 4. Robust branch tracer by natural continuation in R0
# =======================================================================
def bracket_near(C_prev, mu_N_of_C):
    for lo_f, hi_f in [(0.6, 1.4), (0.3, 2.2), (0.1, 4.0), (0.02, 10.0), (0.003, 30.0)]:
        lo = max(1.0, C_prev * lo_f)
        hi = min(P["K"] * 0.99995, C_prev * hi_f)
        if lo >= hi:
            continue
        glo, ghi = G(lo, mu_N_of_C), G(hi, mu_N_of_C)
        if np.isfinite(glo) and np.isfinite(ghi) and glo * ghi < 0:
            return brentq(G, lo, hi, args=(mu_N_of_C,), xtol=1e-1)
    return None


def global_bracket(mu_N_of_C):
    Cs = np.geomspace(1e3, P["K"] * 0.99995, 300)
    vals = np.array([G(c, mu_N_of_C) for c in Cs])
    for i in range(len(Cs) - 1):
        if np.isfinite(vals[i]) and np.isfinite(vals[i + 1]) and vals[i] * vals[i + 1] < 0:
            return brentq(G, Cs[i], Cs[i + 1], args=(mu_N_of_C,), xtol=1e-1)
    return None


def trace_upper_branch(mu_N_maker, R0_hi=2.5, R0_lo=0.5, n=500):
    R0s = np.linspace(R0_hi, R0_lo, n)
    Cs = np.full(n, np.nan)
    mu_N0 = mu_N_maker(R0s[0])
    C_prev = global_bracket(mu_N0)
    Cs[0] = C_prev
    R0_fold = R0_lo
    for i in range(1, n):
        mu_N_of_C = mu_N_maker(R0s[i])
        found = bracket_near(C_prev, mu_N_of_C)
        if found is None:
            R0_fold = R0s[i - 1]
            break
        Cs[i] = found
        C_prev = found
    return R0s, Cs, R0_fold


def trace_lower_branch(mu_N_maker, R0_fold, R0_hi=1.0, n=200):
    R0_seed = R0_fold + 1e-3
    mu_N0 = mu_N_maker(R0_seed)
    Cs_scan = np.geomspace(1e3, P["K"] * 0.5, 4000)
    vals = np.array([G(c, mu_N0) for c in Cs_scan])
    small_root = None
    for i in range(len(Cs_scan) - 1):
        if np.isfinite(vals[i]) and np.isfinite(vals[i + 1]) and vals[i] * vals[i + 1] < 0:
            small_root = brentq(G, Cs_scan[i], Cs_scan[i + 1], args=(mu_N0,), xtol=1e-1)
            break
    if small_root is None:
        return np.array([]), np.array([])
    R0s = np.linspace(R0_seed, R0_hi, n)
    Cs = np.full(n, np.nan)
    Cs[0] = small_root
    C_prev = small_root
    for i in range(1, n):
        mu_N_of_C = mu_N_maker(R0s[i])
        found = bracket_near(C_prev, mu_N_of_C)
        if found is None or found > P["K"] * 0.3:
            break
        Cs[i] = found
        C_prev = found
    return R0s, Cs


# =======================================================================
# 5. Trace all four branches
#
# For Model 2, delta2 (antigenic threshold for ACT proliferation) is
# raised from the Table 2.1 point estimate (5e7) to 3.0e8 -- still well
# within its literature range [1e7, 6.7e8] -- for BOTH the forward and
# backward regimes, so that only Gamma_max differs between them (mirrors
# Model 1, where only N_M1 differs). At the Table 2.1 point estimate,
# Model 2's forward branch is numerically genuine but stays two orders of
# magnitude smaller than Model 1's, which was not an informative visual
# pairing; delta2=3e8 was the largest value found (by systematic scan)
# for which NO hidden fold appears anywhere in R0 in [0.5,2.5] for the
# forward case, giving comparable scale to Model 1 without introducing
# spurious bistability.
# =======================================================================
print("Model 1 -- backward (N_M1=2.7e5) ...")
mk_m1_back = lambda R0: make_mu_N_M1(R0, 2.7e5)
R0u_1b, Cu_1b, fold_1b = trace_upper_branch(mk_m1_back)
R0l_1b, Cl_1b = trace_lower_branch(mk_m1_back, fold_1b)
print(f"  R0c = {fold_1b:.4f}")

print("Model 2 -- backward (Gamma_max=2.0e-5, delta2=3.0e8) ...")
Gamma_back = make_Gamma_fun(2.0e-5)
mk_m2_back = lambda R0: make_mu_N_M2(R0, Gamma_back, delta2_val=3.0e8)
R0u_2b, Cu_2b, fold_2b = trace_upper_branch(mk_m2_back)
R0l_2b, Cl_2b = trace_lower_branch(mk_m2_back, fold_2b)
print(f"  R0c = {fold_2b:.4f}")

# =======================================================================
# 6. Figure: 2 panels (Model 1 backward, Model 2 backward), each with a
#    zoomed inset on the bistability window around the fold.
# =======================================================================
from mpl_toolkits.axes_grid1.inset_locator import mark_inset

plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "font.family": "DejaVu Sans", "savefig.facecolor": "white", "savefig.dpi": 200,
    "axes.grid": True, "grid.color": "#EEEEEE", "grid.linewidth": 0.5,
})

COL_DFE = "#1565C0"
COL_ENDEMIC = "#C62828"
YMAX = 4.2

fig, axes = plt.subplots(1, 2, figsize=(14.5, 6.3))

panels = [
    (axes[0], "a) Model 1 (unstructured model) -- backward bifurcation", R0u_1b, Cu_1b, R0l_1b, Cl_1b, fold_1b),
    (axes[1], "b) Model 2 (structured model) -- backward bifurcation", R0u_2b, Cu_2b, R0l_2b, Cl_2b, fold_2b),
]

for panel_idx, (ax, title, R0u, Cu, R0l, Cl, fold) in enumerate(panels):
    R0_grid = np.linspace(0.3, 2.5, 400)
    mask_dfe_stable = R0_grid <= 1
    h1, = ax.plot(R0_grid[mask_dfe_stable], np.zeros(mask_dfe_stable.sum()), "-", color=COL_DFE, lw=3.0)
    h2, = ax.plot(R0_grid[~mask_dfe_stable], np.zeros((~mask_dfe_stable).sum()), "--", color=COL_DFE, lw=2.2)
    h3, = ax.plot(R0u, Cu / 1e8, "-", color=COL_ENDEMIC, lw=3.0)
    h4, = ax.plot(R0l, Cl / 1e8, "--", color=COL_ENDEMIC, lw=2.2)

    ax.axvline(1.0, color="0.55", lw=1.2, ls=":", zorder=2)
    ax.plot(1.0, 0.0, "o", color="black", ms=9, zorder=9)

    ax.set_xlabel(rf"$\mathcal{{R}}_0^{{({panel_idx + 1})}}$", fontsize=14.5)
    ax.set_ylabel(r"$C^*$ ($\times10^{8}$ tumour cells)", fontsize=13)
    ax.set_title(title, fontsize=15, fontweight="bold")
    ax.set_xlim(0.3, 2.5)
    ax.set_ylim(-0.15, YMAX)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(labelsize=11)
    if panel_idx == 0:   # legend on ONE panel only
        ax.legend([h1, h2, h3, h4],
                  ["DFE stable ($C^*=0$)", "DFE unstable",
                   "Endemic equilibrium stable", "Endemic equilibrium unstable"],
                  loc="upper left", fontsize=10, framealpha=0.93, edgecolor="#CCCCCC")

fig.tight_layout()
fig.savefig("./fig2_backward_bifurcation_final_nouveau_joel_en.png", bbox_inches="tight", dpi=200)
print("Saved fig2_backward_bifurcation_final_nouveau_joel_en.png")